# 07m — Busca de hiperparâmetros do Random Forest multiclasse (grade completa, validação cruzada de 5 dobras e sementes)

Este notebook responde: **quais hiperparâmetros do RF multiclasse (7 classes, sem os atributos de AUC alto) funcionam melhor, e quanto cada um influencia o resultado?** Depois treina o modelo final com a combinação escolhida, com o mesmo protocolo e o mesmo cálculo de custo do notebook 05m. A avaliação no teste fica para o notebook 08m.

| O que | Como |
|---|---|
| **Grade completa** | `n_estimators` × `max_depth` × `min_samples_leaf` (3 × 3 × 3 = **27 combinações**, as mesmas do binário), todas cruzadas. Valores em `GRADE` |
| **Pesos de classe** | `class_weight="balanced"` em todas as combinações (decisão do protocolo), recalculado pelo RF em cada treino |
| **Atalho das árvores** | Com a mesma semente, as primeiras *k* árvores de uma floresta maior são **idênticas** às de uma floresta de *k* árvores. Treina-se uma floresta com o maior `n_estimators` e avaliam-se as primeiras 100, 300 e 500. O notebook confere isso na sua versão do scikit-learn |
| **Validação cruzada** | 5 dobras no **ajuste** (239.671 linhas), estratificadas por classe. As dobras são **as mesmas** para todas as combinações (comparação pareada) e serão as mesmas do SVM |
| **Sementes** | 3 sementes do RF por combinação, para medir o ruído do sorteio do modelo |
| **Validação separada** | Cada combinação também é treinada no ajuste inteiro e avaliada na **validação** (59.918 linhas), sem pesos e na proporção do treino |
| **Métricas** | ROC-AUC um-contra-resto (média macro), acurácia balanceada, acurácia, F1 macro e MCC multiclasse, **mais o recall de cada classe** |
| **Custo** | Para cada combinação: nós, profundidade, tamanho do modelo, tempo de treino e de previsão. Para o modelo final: o mesmo custo completo do notebook 05m (tempo, CPU, RAM, latência, vazão) |

## Regra de escolha (definida **antes** de ver os resultados)
1. Só a **validação cruzada** decide (o teste não entra, e só é usado uma vez, no notebook 08m).
2. Duas métricas de decisão: **ROC-AUC um-contra-resto (macro)** e **MCC multiclasse**, médias das dobras (e das sementes).
3. **Regra de um erro-padrão:** para cada métrica, toda combinação a **menos de 1 erro-padrão** da melhor conta como empatada com ela. Fica elegível quem empata **nas duas** métricas.
4. Entre as elegíveis, vence a **mais barata** (menos nós no total, que reflete tempo e memória).
5. Se nenhuma for elegível nas duas, vence a de melhor posição média nas duas métricas.

O **recall por classe** é só **relatado** (não decide). Se a escolhida piorar muito o recall de alguma classe em relação ao padrão, isso aparece na tabela da seção 7 e deve ser discutido no TCC.

## O que o notebook grava
Uma pasta `artefatos_modelos/rf_multiclasse_busca/rfm_busca_<data>` com: os resultados de toda a grade (CSV), os efeitos de cada hiperparâmetro (CSV e gráficos), a combinação escolhida e por quê, o modelo do ajuste e o **modelo final (refit)**, as métricas da validação, o recall por classe e o custo computacional.

## Antes de rodar
- Os notebooks 02, 03m, 04m e **05m** precisam ter terminado (este usa as mesmas views e o mesmo refit).
- Feche os outros notebooks (cada kernel aberto guarda memória) e deixe o computador na tomada, sem suspensão.
- A busca é **longa**: 9 pares (profundidade × folha) × 3 sementes × (5 dobras + 1 treino completo) = **162 florestas de 500 árvores**. O treino de 100 árvores levou ~40 s no 05m; florestas de 500 árvores sem limite de profundidade levam alguns minutos, então estime **várias horas** (a barra de progresso mostra o andamento e o tempo restante). Cada unidade pronta é salva: se algo interromper, **rode a célula de novo** e ela retoma de onde parou.
- **Memória:** o RF multiclasse sem limite de profundidade é grande (o modelo do 05m, com 100 árvores, tem ~3,5 milhões de nós e 67 MiB em disco). A floresta de 500 árvores sem limite de profundidade deve ocupar **da ordem de 2,5 a 3 GiB** durante o treino. A célula 3.1 mostra a memória livre e avisa se pode faltar; se faltar, reduza o maior `n_estimators` da grade (por exemplo, `(100, 200, 300)`).
- **Se a busca já estiver FINALIZADA** (mesma configuração e mesmos dados), o notebook entra em **modo leitura**: carrega o modelo e os custos gravados, refaz só os cálculos baratos, confere tudo e **não retreina nem sobrescreve nada**. Para refazer de propósito, defina `NOVA_EXECUCAO = True`; para reaproveitar uma pasta específica (com verificação dos dados contra o 05m), `FORCAR_PASTA = r"...\rfm_busca_<data>"`, ambos antes da célula 1.1.
- Para encurtar: `SEMENTES = (42,)` ou `K_CV = 3` reduzem o tempo, ao custo de menos precisão no erro-padrão.
- Requisitos: `scikit-learn`, `psutil` e `matplotlib` (opcional, para os gráficos).

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse.afonsolelis.tech:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# ---------- métricas multiclasse (a partir da matriz de confusão; aceitam pesos por linha) ----------
def metricas_multi(y, proba, w=None):
    y = np.asarray(y, dtype=int); proba = np.asarray(proba, dtype=float); K_ = proba.shape[1]; pred = proba.argmax(axis=1); w_ = None if w is None else np.asarray(w, dtype=float)
    cm = np.zeros((K_, K_)); np.add.at(cm, (y, pred), 1.0 if w_ is None else w_)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)                  # total, linhas por classe verdadeira, previstas por classe, acertos
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k).astype(int)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, proba[:, k], sample_weight=w_)
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t}

# autoteste das ferramentas (não toca no servidor): confere as métricas multiclasse contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc, precision_score as _prec
_r = np.random.default_rng(0); _y = _r.integers(0, 4, 400); _P = _r.dirichlet(np.ones(4), 400); _P[np.arange(400), _y] += 0.35; _P /= _P.sum(axis=1, keepdims=True); _w = _r.uniform(0.5, 3, 400); _pr = _P.argmax(axis=1)
for _ww in (None, _w):
    _e, _a = metricas_multi(_y, _P, _ww)
    assert abs(_e["acuracia"] - _acc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr, sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["f1_macro"] - _f1(_y, _pr, average="macro", sample_weight=_ww)) < 1e-12 and abs(_e["f1_ponderado"] - _f1(_y, _pr, average="weighted", sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["mcc"] - _mcc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["precisao_macro"] - _prec(_y, _pr, average="macro", sample_weight=_ww, zero_division=0)) < 1e-12, _e
    assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P, multi_class="ovr", average="macro", sample_weight=_ww)) < 1e-12, _e
assert abs(metricas_multi(np.array([0, 1, 2]), np.eye(3))[0]["mcc"] - 1.0) < 1e-12
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


### Ferramentas da busca
- **`prever_prefixos`**: probabilidades das classes usando as primeiras *k* árvores de uma floresta (o atalho das árvores).
- **`recall_por_classe`** e **`met`**: o recall de cada classe e as métricas multiclasse do notebook 05m.
- **`Barra`**: barra de progresso com tempo restante estimado.

In [6]:
import copy, io, gc

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")

import warnings
def silenciar_aviso_sklearn():
    # Com muitos núcleos, o scikit-learn às vezes deixa a lista de filtros de aviso VAZIA (condição de corrida entre threads) e passa a emitir, a cada árvore, um aviso inofensivo
    # sobre `delayed`/`Parallel`. Duas proteções: (1) um filtro que ignora essa mensagem; (2) um desvio em `warnings.warn` que descarta SÓ essa mensagem, mesmo que a lista de
    # filtros seja esvaziada de novo durante uma previsão. Os outros avisos continuam funcionando. Não muda nenhum resultado.
    warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*", category=UserWarning)
    if not getattr(warnings.warn, "_filtrado_sklearn", False):
        _orig = warnings.warn
        def _warn_filtrado(message, *args, **kwargs):
            if isinstance(message, str) and message.startswith("`sklearn.utils.parallel.delayed` should be used with"): return
            return _orig(message, *args, **kwargs)
        _warn_filtrado._filtrado_sklearn = True; warnings.warn = _warn_filtrado
silenciar_aviso_sklearn()

def _sub_floresta(rf, ini, fim):
    sub = copy.copy(rf); sub.estimators_ = rf.estimators_[ini:fim]; sub.n_estimators = fim - ini; return sub

def prever_prefixos(rf, X, tamanhos):
    # Probabilidades das K classes usando as primeiras k árvores, para cada k de `tamanhos` (crescente), numa só passada pelas árvores.
    # Devolve ({k: matriz n x K float32}, {k: tempo acumulado de previsão em s}).
    silenciar_aviso_sklearn()
    X32 = np.ascontiguousarray(X, dtype=np.float32); soma = np.zeros((len(X32), len(rf.classes_))); saida, tempos, ant, acum = {}, {}, 0, 0.0
    for k in tamanhos:
        t0 = time.perf_counter(); p = _sub_floresta(rf, ant, k).predict_proba(X32); acum += time.perf_counter() - t0
        soma += p * (k - ant); saida[k] = (soma / k).astype(np.float32); tempos[k] = acum; ant = k
    return saida, tempos

def recall_por_classe(y, P):
    # Recall de cada classe (a classe prevista é a de maior probabilidade).
    y = np.asarray(y, dtype=int); K_ = P.shape[1]; pred = np.asarray(P).argmax(axis=1); cm_ = np.bincount(y * K_ + pred, minlength=K_ * K_).reshape(K_, K_).astype(float)
    return np.divide(np.diag(cm_), cm_.sum(axis=1), out=np.full(K_, np.nan), where=cm_.sum(axis=1) > 0)

def met(y, P, w=None):
    # Métricas multiclasse do notebook 05m, com o ROC-AUC um-contra-resto (macro) chamado de `roc_auc`.
    e, a = metricas_multi(y, P, w); e = dict(e); e["roc_auc"] = e.pop("roc_auc_ovr_macro"); return e, a

# autoteste (não toca no servidor): barra, prefixos, recall por classe
_rng = np.random.default_rng(0); _rf = RandomForestClassifier(n_estimators=30, random_state=1, n_jobs=1).fit(_rng.normal(size=(300, 4)), _rng.integers(0, 3, 300))
_X = np.random.default_rng(2).normal(size=(50, 4)); _pf, _ = prever_prefixos(_rf, _X, [10, 30])
assert np.allclose(_pf[30], _rf.predict_proba(_X.astype(np.float32)), atol=1e-6), "prever_prefixos não reproduz o predict_proba"
_P = np.array([[.8, .1, .1], [.2, .7, .1], [.1, .2, .7], [.6, .3, .1]]); _r = recall_por_classe(np.array([0, 1, 2, 1]), _P); assert np.allclose(_r, [1.0, 0.5, 1.0]), _r
_b = Barra(2, "autoteste"); _b.avancar(2); assert fmt_tempo(3725) == "1h 02min"
print("Autoteste das ferramentas da busca OK")

Autoteste das ferramentas da busca OK


In [7]:
   FORCAR_PASTA = r"C:\Users\Bruno\Downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_multiclasse_busca\rfm_busca_20261102_001711"

## 1. Parâmetros
Tudo o que define a busca fica aqui: a grade, as sementes, as dobras e a regra de escolha. **Não mude nada depois de ver resultados.**

In [8]:
# ---------- 1.1 Parâmetros ----------
GRADE = globals().get("GRADE", {"n_estimators": (100, 300, 500), "max_depth": (None, 15, 8), "min_samples_leaf": (1, 5, 10)})
SEMENTES = tuple(globals().get("SEMENTES", (42, 43, 44)))             # sementes do RF (a aleatoriedade do modelo)
SEMENTE = globals().get("SEMENTE", 42)                                # dobras da validação cruzada, partições da validação, refit e amostras de latência
K_CV = globals().get("K_CV", 5)
K_PARTICOES_VAL = globals().get("K_PARTICOES_VAL", 5)
REPETICOES_TEMPO = globals().get("REPETICOES_TEMPO", 3)
N_LATENCIA = globals().get("N_LATENCIA", 1000)
COMPRESSAO_MODELO = globals().get("COMPRESSAO_MODELO", 3)
TAMANHOS_CURVA = tuple(globals().get("TAMANHOS_CURVA", (25, 50, 100, 200, 300, 400, 500)))   # tamanhos extras (gratuitos) para a curva "desempenho × número de árvores"
MULT_ERRO_PADRAO = globals().get("MULT_ERRO_PADRAO", 1.0)
METRICAS_DECISAO = ("roc_auc", "mcc")                                  # ROC-AUC um-contra-resto (macro) e MCC multiclasse
PADRAO = {"n_estimators": 100, "max_depth": None, "min_samples_leaf": 1}      # a configuração do notebook 05m
PARAR_APOS_UNIDADES = globals().get("PARAR_APOS_UNIDADES", None)             # só para testes
NOVA_EXECUCAO = globals().get("NOVA_EXECUCAO", False)                        # True = não retomar uma busca interrompida
FIXOS = dict(min_samples_split=2, max_features="sqrt", class_weight="balanced", n_jobs=-1)
assert set(GRADE) == {"n_estimators", "max_depth", "min_samples_leaf"} and all(len(v) >= 2 for v in GRADE.values()), "GRADE precisa ter os 3 hiperparâmetros, cada um com 2 valores ou mais"

TAM_GRADE = sorted(GRADE["n_estimators"]); NMAX = max(TAM_GRADE)
TAM_CURVA_USADOS = tuple(k for k in TAMANHOS_CURVA if k <= NMAX); TAM_TODOS = sorted(set(TAM_GRADE) | set(TAM_CURVA_USADOS))
rot = lambda d: "None" if d is None else str(d)
nome_unidade = lambda d, l, s: f"prof{rot(d)}_folha{l}_semente{s}"
PARES = [(d, l) for d in GRADE["max_depth"] for l in GRADE["min_samples_leaf"]]
UNIDADES = [(d, l, s) for (d, l) in PARES for s in SEMENTES]
CONFIGS = [(n, d, l) for n in TAM_GRADE for (d, l) in PARES]
N_FLORESTAS = len(UNIDADES) * (K_CV + 1)
ARTEFATOS_BASE = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "rf_multiclasse_busca"; ARTEFATOS_BASE.mkdir(parents=True, exist_ok=True)

V_AJUSTE, V_VALID, V_REFIT = "prep_vw_bal_multi_treino", "prep_vw_bal_multi_validacao", "prep_vw_bal_multi_refit"
SEL_TAB, DADOS_TAB, TAB_REFIT = "prep_bal_multi_selecao", "prep_bal_multi_dados", "prep_bal_multi_refit_sel"
AUD_TAB, TAB_CLASSES = "prep_views_rf_multi", "prep_classes_rf_multi"
faltam = [t for t in ("prep_final_treino", V_AJUSTE, V_VALID, V_REFIT, SEL_TAB, DADOS_TAB, AUD_TAB, TAB_CLASSES, TAB_REFIT) if not existe(t)]
assert not faltam, f"Rode antes os notebooks 02, 03m, 04m e 05m: faltam {faltam}"

versao_sel = str(consultar(f"SELECT any(versao) AS v FROM {SEL_TAB}")["v"][0])
aud = consultar_num(f"SELECT posicao, feature, situacao, mediana_imputacao, tem_indicador, versao FROM {AUD_TAB} ORDER BY posicao"); versao_views = str(aud["versao"].iloc[0])
cl = consultar_num(f"SELECT codigo, classe, linhas_treino FROM {TAB_CLASSES} ORDER BY codigo")
CLASSES = cl.classe.tolist(); K = len(CLASSES); COD = {c: int(k) for c, k in zip(cl.classe, cl.codigo)}; N_TREINO = {c: int(n) for c, n in zip(cl.classe, cl.linhas_treino)}
assert [COD[c] for c in CLASSES] == list(range(K)), "os códigos das classes precisam ser 0..K-1"
cont = consultar_num(f"SELECT papel, y, count() AS n FROM {SEL_TAB} GROUP BY papel, y"); cont = {(r.papel, int(r.y)): int(r.n) for r in cont.itertuples()}
N_AJ = {c: cont[("treino", COD[c])] for c in CLASSES}; N_VA = {c: cont[("validacao", COD[c])] for c in CLASSES}
print(f"Seleção do notebook 04m (versão {versao_sel}) | views do notebook 03m (versão {versao_views})")
print(f"{K} classes: {CLASSES}")
print(pd.DataFrame({"no treino": N_TREINO, "ajuste": N_AJ, "validação": N_VA}).T.to_string())
print(f"\nGrade: {len(CONFIGS)} configurações = {len(TAM_GRADE)} (n_estimators {TAM_GRADE}) × {len(GRADE['max_depth'])} (max_depth {[rot(d) for d in GRADE['max_depth']]}) × {len(GRADE['min_samples_leaf'])} (min_samples_leaf {list(GRADE['min_samples_leaf'])})")
print(f"Sementes: {SEMENTES} | validação cruzada: {K_CV} dobras | class_weight = {FIXOS['class_weight']}")
print(f"Treinos: {len(PARES)} pares (profundidade × folha) × {len(SEMENTES)} sementes × ({K_CV} dobras + 1 treino completo) = {N_FLORESTAS} florestas de {NMAX} árvores")
print(f"(as florestas de {', '.join(str(k) for k in TAM_GRADE if k < NMAX)} árvores saem das primeiras árvores de cada floresta de {NMAX})")

Seleção do notebook 04m (versão 20261027_173009) | views do notebook 03m (versão 20261026_010742)
7 classes: ['BENIGN', 'LDAP', 'MSSQL', 'NetBIOS', 'Syn', 'UDP', 'UDPLag']
           BENIGN   LDAP   MSSQL  NetBIOS     Syn      UDP  UDPLag
no treino   50768  30099  207003    19490  155515  1075337   89066
ajuste      40000  24079   40000    15592   40000    40000   40000
validação   10000   6020   10000     3898   10000    10000   10000

Grade: 27 configurações = 3 (n_estimators [100, 300, 500]) × 3 (max_depth ['None', '15', '8']) × 3 (min_samples_leaf [1, 5, 10])
Sementes: (42, 43, 44) | validação cruzada: 5 dobras | class_weight = balanced
Treinos: 9 pares (profundidade × folha) × 3 sementes × (5 dobras + 1 treino completo) = 162 florestas de 500 árvores
(as florestas de 100, 300 árvores saem das primeiras árvores de cada floresta de 500)


## 2. Dados, pré-processamento, classes e dobras
Lê o ajuste, a validação e o refit (as mesmas views do notebook 05m), **ordena por `id_linha`** (para as dobras serem reproduzíveis e iguais às do SVM) e confere que o `PreprocessadorRF` reproduz a matriz das views. As **dobras** da validação cruzada são estratificadas por classe e valem para todas as combinações.

A busca guarda tudo em uma pasta cuja **impressão digital** depende da grade, das sementes, das dobras e dos dados: se a busca for interrompida, rodar de novo **retoma** na mesma pasta; se algo mudar, começa uma nova.

In [9]:
# ---------- 2.1 Dados, pré-processamento, classes e dobras ----------
import hashlib
USADAS, RETIRADAS = aud[aud.situacao == "usada"], aud[aud.situacao != "usada"].feature.tolist()
MEDIANAS = {r.feature: float(r.mediana_imputacao) for r in USADAS.itertuples() if int(r.tem_indicador) == 1}
prep = PreprocessadorRF(USADAS.feature.tolist(), MEDIANAS, RETIRADAS, versao_views)
assert prep.colunas_modelo == [c for c in colunas_view(V_AJUSTE) if c not in ("id_linha", "y")], "a ordem das colunas da view difere da do pré-processamento"
print(f"Atributos do modelo: {len(prep.colunas_modelo)} ({len(prep.features)} + {len(prep.indicadores)} indicadores) | retiradas na ablação: {RETIRADAS}")

def ordenar(df): return df.iloc[np.argsort(df["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True)
aj, va, rf_df = ordenar(ler_view(V_AJUSTE)), ordenar(ler_view(V_VALID)), ordenar(ler_view(V_REFIT))
CM = prep.colunas_modelo
X_aj, y_aj = aj[CM].to_numpy(float), aj["y"].to_numpy(int)
X_va, y_va, w_va = va[CM].to_numpy(float), va["y"].to_numpy(int), va["peso"].to_numpy(float)
X_rf, y_rf = rf_df[CM].to_numpy(float), rf_df["y"].to_numpy(int)
for nome, X in (("ajuste", X_aj), ("validação", X_va), ("refit", X_rf)): assert np.isfinite(X).all(), f"valores não finitos em {nome}"
X_va_raw = X_va[:, :len(prep.features)].copy()
for j, c in enumerate(prep.indicadores): X_va_raw[X_va[:, len(prep.features) + j] == 1, prep.features.index(c)] = np.nan
assert np.allclose(prep.transformar(X_va_raw), X_va, rtol=1e-12, atol=1e-12), "o PreprocessadorRF não reproduz a matriz da view"
conta = lambda y: {CLASSES[k]: int((y == k).sum()) for k in range(K)}
assert conta(y_aj) == N_AJ and conta(y_va) == N_VA, "as linhas por classe não coincidem com a seleção do notebook 04m"
g_aj, g_va = y_aj, y_va                                                    # no multiclasse, y já é o código da classe

DOBRAS = list(StratifiedKFold(K_CV, shuffle=True, random_state=SEMENTE).split(X_aj, y_aj))
assert all(len(np.unique(y_aj[ite])) == K for _, ite in DOBRAS), "alguma dobra ficou sem alguma classe"
tab_classes = pd.DataFrame({"ajuste": np.bincount(y_aj, minlength=K), "validação": np.bincount(y_va, minlength=K)}, index=CLASSES)
print(f"\nAjuste: {len(X_aj):,} linhas | validação: {len(X_va):,} | refit: {len(X_rf):,} | {K} classes")
display(tab_classes.T)
print(f"Dobras: {K_CV}, estratificadas por classe (cada dobra de avaliação tem ~{len(X_aj) // K_CV:,} linhas)")

# pasta da execução: retoma uma busca interrompida com a MESMA configuração e os MESMOS dados
crc = lambda s: zlib.crc32(",".join(s).encode())
IMPRESSAO = hashlib.sha1(json.dumps(_limpar({"grade": GRADE, "sementes": SEMENTES, "semente": SEMENTE, "k_cv": K_CV, "tam_curva": TAM_CURVA_USADOS, "fixos": FIXOS, "versao_sel": versao_sel,
                                           "versao_views": versao_views, "ids_aj": crc(aj["id_linha"]), "ids_va": crc(va["id_linha"])}), sort_keys=True).encode()).hexdigest()[:12]
_a = np.zeros(len(y_aj), np.int8)
for k_, (_, ite_) in enumerate(DOBRAS): _a[ite_] = k_
IMPRESSAO_DOBRAS = format(zlib.crc32(_a.tobytes()), "08x")                 # identifica as dobras: o SVM multiclasse deve imprimir o mesmo código
PASTA = None; FORCAR_PASTA = globals().get("FORCAR_PASTA", None); FORCADA = False; MODO_LEITURA = False      # MODO_LEITURA: busca já FINALIZADA, reaproveitada sem retreinar nem sobrescrever
COMPONENTES = {"versao_selecao": versao_sel, "versao_views": versao_views, "crc_ids_ajuste": crc(aj["id_linha"]), "crc_ids_validacao": crc(va["id_linha"]), "k_cv": K_CV, "semente": SEMENTE, "tam_curva": list(TAM_CURVA_USADOS)}
print("Componentes da impressão digital da busca:", COMPONENTES)
anteriores, finalizadas = [], []                                      # incompletas: (pasta, impressão, nº de unidades prontas) | finalizadas: (pasta, impressão, mesma grade e sementes?)
for p in sorted(ARTEFATOS_BASE.glob("rfm_busca_*"), reverse=True):
    try: cfgp = json.loads((p / "config_busca.json").read_text(encoding="utf-8"))
    except (OSError, ValueError): continue
    if (p / "FINALIZADO").exists(): finalizadas.append((p, cfgp.get("impressao"), cfgp.get("grade") == _limpar(GRADE) and cfgp.get("sementes") == _limpar(SEMENTES)))
    else: anteriores.append((p, cfgp.get("impressao"), len(list((p / "unidades").glob("*.npz")))))
if FORCAR_PASTA:                                                      # retomada FORÇADA de uma pasta específica, só depois de verificar que os dados atuais são os mesmos do 05m
    PASTA = Path(FORCAR_PASTA); assert (PASTA / "unidades").exists() and (PASTA / "config_busca.json").exists(), f"{PASTA} não é uma pasta de busca (faltam unidades ou config_busca.json)"
    _p05 = globals().get("PASTA_05M") or next(iter(sorted((p for p in (ARTEFATOS_BASE.parent / "rf_multiclasse").glob("rfm_*") if all((p / f).exists() for f in ("refit_ids.csv", "preprocessamento.json", "classes.json"))), reverse=True)), None)
    assert _p05, "não achei a pasta do notebook 05m (rf_multiclasse\\rfm_*) para conferir os dados: defina PASTA_05M"
    _rid = pd.read_csv(Path(_p05) / "refit_ids.csv", dtype={"id_linha": str}); _pc = json.loads((Path(_p05) / "preprocessamento.json").read_text(encoding="utf-8")); _cl = json.loads((Path(_p05) / "classes.json").read_text(encoding="utf-8"))
    _m_aj = aj[["id_linha", "y"]].merge(_rid[_rid.origem == "ajuste"][["id_linha", "y"]], on="id_linha", suffixes=("", "_05")); _m_va = va[["id_linha", "y"]].merge(_rid[_rid.origem == "validacao"][["id_linha", "y"]], on="id_linha", suffixes=("", "_05"))
    _z0 = np.load(next((PASTA / "unidades").glob("*.npz")))["oof"].shape if any((PASTA / "unidades").glob("*.npz")) else None
    verif = {"linhas do ajuste = as do 05m (mesmos ids e mesma classe)": len(_m_aj) == len(aj) == int((_rid.origem == "ajuste").sum()) and bool((_m_aj.y == _m_aj.y_05).all()),
             "linhas da validação = as do 05m (mesmos ids e mesma classe)": len(_m_va) == len(va) == int((_rid.origem == "validacao").sum()) and bool((_m_va.y == _m_va.y_05).all()),
             "atributos e ordem das colunas = os do 05m": _pc["colunas_modelo"] == CM, "medianas de imputação = as do 05m": _pc["medianas_treino"] == MEDIANAS,
             "classes = as do 05m": _cl["classes"] == CLASSES, "formato das unidades = o dos dados atuais": _z0 is None or _z0 == (len(TAM_GRADE), len(y_aj), K)}
    for k_, v_ in verif.items(): print(("✔" if v_ else "✘"), k_)
    assert all(verif.values()), "OS DADOS ATUAIS NÃO COINCIDEM COM OS DA BUSCA ANTERIOR: NÃO reutilize essas unidades (a busca precisa ser refeita)"
    FORCADA = True; MODO_LEITURA = (PASTA / "FINALIZADO").exists()
    print(f"✔ Retomada FORÇADA de {PASTA.name}: os dados atuais são os mesmos usados no 05m ({Path(_p05).name})")
elif not NOVA_EXECUCAO:
    for p, imp, n_u in anteriores:
        if imp == IMPRESSAO: PASTA = p; break
    ja_feitas = [(p, imp) for p, imp, mesma in finalizadas if mesma]; iguais = [p for p, imp in ja_feitas if imp == IMPRESSAO]
    if PASTA is None and iguais: PASTA = iguais[0]; MODO_LEITURA = True       # busca FINALIZADA com a mesma configuração e os mesmos dados: reaproveita (nada será retreinado nem sobrescrito)
    elif PASTA is None and ja_feitas:                                 # mesma grade, mas dados ou versões diferentes: não decide sozinho
        raise RuntimeError("Existe uma busca FINALIZADA com esta mesma grade e sementes, mas com impressão digital DIFERENTE: " + "; ".join(f"{p.name} (impressão {imp})" for p, imp in ja_feitas) + f". Impressão atual: {IMPRESSAO}. "
                           "Os dados ou as versões mudaram desde aquela busca: confira os componentes impressos acima. Para reaproveitar mesmo assim essa pasta (com verificação dos dados contra o 05m), defina FORCAR_PASTA = r'<caminho>' antes desta célula. "
                           "Para refazer a busca DE PROPÓSITO, defina NOVA_EXECUCAO = True.")
    if PASTA is None and any(n_u > 0 for _, _, n_u in anteriores):   # há busca anterior incompleta que NÃO coincide: não recomeça sozinho (evita refazer horas de trabalho por engano)
        raise RuntimeError("Existe(m) busca(s) anterior(es) incompleta(s) que NÃO coincide(m) com a configuração e os dados atuais:\n" + "\n".join(f"  - {p.name}: {n_u} unidades prontas (impressão {imp})" for p, imp, n_u in anteriores)
                           + f"\nImpressão digital atual: {IMPRESSAO}. Se você esperava RETOMAR uma delas (por exemplo, depois de o servidor voltar), confira as 'Componentes' impressas acima (versões da seleção e das views, ids) com as da execução anterior. "
                           "Para retomar uma pasta específica, com verificação dos dados, defina FORCAR_PASTA = r'<caminho da pasta rfm_busca_...>' antes desta célula. Para começar uma busca NOVA de propósito, defina NOVA_EXECUCAO = True.")
RETOMANDO = PASTA is not None
if PASTA is None:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S"); PASTA = ARTEFATOS_BASE / f"rfm_busca_{RUN_ID}"; PASTA.mkdir(parents=True)
    salvar_json({"run_id": RUN_ID, "criado_em": datetime.now().isoformat(timespec="seconds"), "impressao": IMPRESSAO, "componentes": COMPONENTES, "grade": GRADE, "sementes": SEMENTES}, PASTA / "config_busca.json")
else:
    RUN_ID = PASTA.name.replace("rfm_busca_", "")
    if FORCADA and not MODO_LEITURA:
        _c = json.loads((PASTA / "config_busca.json").read_text(encoding="utf-8")); _c.update({"impressao_original": _c.get("impressao"), "impressao": IMPRESSAO, "componentes": COMPONENTES,
            "retomada_forcada_em": datetime.now().isoformat(timespec="seconds"), "verificacao_da_retomada_forcada": verif}); salvar_json(_c, PASTA / "config_busca.json")
(PASTA / "unidades").mkdir(exist_ok=True)
print(f"\nImpressão digital das dobras: {IMPRESSAO_DOBRAS} (o SVM multiclasse precisa dar o mesmo código)")
print(f"Pasta da execução: {PASTA}" + ("  (MODO LEITURA: busca FINALIZADA reaproveitada; nada será retreinado nem sobrescrito)" if MODO_LEITURA else ("  (RETOMANDO uma busca interrompida)" if RETOMANDO else "")))

Atributos do modelo: 44 (42 + 2 indicadores) | retiradas na ablação: ['bwd_packets_s', 'inbound', 'total_backward_packets']

Ajuste: 239,671 linhas | validação: 59,918 | refit: 299,589 | 7 classes


,BENIGN,LDAP,MSSQL,NetBIOS,Syn,UDP,UDPLag
ajuste,40000,24079,40000,15592,40000,40000,40000
validação,10000,6020,10000,3898,10000,10000,10000


Dobras: 5, estratificadas por classe (cada dobra de avaliação tem ~47,934 linhas)
Componentes da impressão digital da busca: {'versao_selecao': '20261027_173009', 'versao_views': '20261026_010742', 'crc_ids_ajuste': 559776259, 'crc_ids_validacao': 3448974085, 'k_cv': 5, 'semente': 42, 'tam_curva': [25, 50, 100, 200, 300, 400, 500]}
✔ linhas do ajuste = as do 05m (mesmos ids e mesma classe)
✔ linhas da validação = as do 05m (mesmos ids e mesma classe)
✔ atributos e ordem das colunas = os do 05m
✔ medianas de imputação = as do 05m
✔ classes = as do 05m
✔ formato das unidades = o dos dados atuais
✔ Retomada FORÇADA de rfm_busca_20261102_001711: os dados atuais são os mesmos usados no 05m (rfm_20261101_210334)

Impressão digital das dobras: 7e97c47f (o SVM multiclasse precisa dar o mesmo código)
Pasta da execução: C:\Users\Bruno\Downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_multiclasse_busca\rfm_busca_20261102_001711  (MODO LEITURA: busca FINALIZADA reaproveitada; nada será retreina

## 3. A busca
Para cada par (profundidade × folha) e cada semente (uma **unidade**), treina uma floresta de `NMAX` árvores em cada uma das 5 dobras e **no ajuste inteiro**, e guarda as probabilidades das 7 classes fora da dobra e na validação para cada `n_estimators` da grade (o atalho das árvores). Cada unidade pronta é salva em `unidades/`: **se interromper, rode a célula de novo**. O tempo de cada sessão fica em `tempos_busca.csv`.

Antes da parte longa, a célula **confere o atalho das árvores** na sua versão do scikit-learn; se ele não valer, a busca não começa.

In [10]:
# ---------- 3.1 Busca ----------
# conferência rápida ANTES da parte longa: as primeiras k árvores de uma floresta maior = floresta treinada com k árvores (na sua versão do scikit-learn)
_i = np.random.default_rng(0).choice(len(X_aj), size=min(3000, len(X_aj)), replace=False)
_g = RandomForestClassifier(n_estimators=40, random_state=7, **FIXOS).fit(X_aj[_i], y_aj[_i]); _p, _ = prever_prefixos(_g, X_va[:2000], [20, 40])
_d = RandomForestClassifier(n_estimators=20, random_state=7, **FIXOS).fit(X_aj[_i], y_aj[_i])
assert np.allclose(_p[20], _d.predict_proba(X_va[:2000]), atol=1e-6), "o atalho das árvores NÃO vale nesta versão do scikit-learn: não rode a busca e avise antes"
print(f"✔ Atalho das árvores conferido (scikit-learn {sklearn.__version__}): as primeiras 20 de 40 árvores = floresta treinada com 20")
del _g, _d, _p
if psutil and None in GRADE["max_depth"]:                         # as florestas sem limite de profundidade são enormes: ~120 bytes por nó (o modelo do 05m, com 100 árvores, tem ~3,5 milhões de nós)
    _disp = psutil.virtual_memory().available / 2 ** 30; _est = 3.5e6 * 120 * (NMAX / 100) * 1.5 / 2 ** 30
    print(f"Memória livre agora: {_disp:.1f} GiB | a maior floresta (profundidade livre, {NMAX} árvores) deve precisar de ~{_est:.1f} GiB no pico (estimativa pelo tamanho do modelo do 05m)")
    if _disp < _est: print(f"⚠ pode faltar memória. Feche outros programas, ou reduza o maior n_estimators (por exemplo GRADE = {{'n_estimators': (100, 200, 300), ...}}) antes de rodar.")
prontas = sum((PASTA / "unidades" / f"{nome_unidade(*u)}.npz").exists() for u in UNIDADES)
print(f"Unidades prontas: {prontas}/{len(UNIDADES)}" + ("  (retomando)" if prontas else "") + f" | florestas a treinar agora: {(len(UNIDADES) - prontas) * (K_CV + 1)}")
auc_ovr = lambda y, P: float(roc_auc_score(y, P, multi_class="ovr", average="macro"))
barra = Barra(N_FLORESTAS, "Busca de hiperparâmetros"); barra.n = prontas * (K_CV + 1); barra._mostrar(forcar=True); novas = 0
t_sessao, inicio_sessao, interrompida = time.perf_counter(), datetime.now().isoformat(timespec="seconds"), True
try:
    for (d, l, s) in UNIDADES:
        arq = PASTA / "unidades" / f"{nome_unidade(d, l, s)}.npz"
        if arq.exists(): continue
        hp = dict(n_estimators=NMAX, max_depth=d, min_samples_leaf=l, random_state=s, **FIXOS); rotulo = f"prof={rot(d)} folha={l} semente={s}"
        oof = np.full((len(TAM_GRADE), len(y_aj), K), np.nan, np.float32); curva_cv = np.zeros((len(DOBRAS), len(TAM_CURVA_USADOS))); t_cv = []
        for f, (itr, ite) in enumerate(DOBRAS):
            silenciar_aviso_sklearn(); t0 = time.perf_counter(); m_ = RandomForestClassifier(**hp).fit(X_aj[itr], y_aj[itr]); t_cv.append(time.perf_counter() - t0)
            pf, _ = prever_prefixos(m_, X_aj[ite], TAM_TODOS)
            for i, k in enumerate(TAM_GRADE): oof[i, ite] = pf[k]
            curva_cv[f] = [auc_ovr(y_aj[ite], pf[k]) for k in TAM_CURVA_USADOS]
            del m_, pf; barra.avancar(1, f"{rotulo} | dobra {f + 1}/{len(DOBRAS)} ({t_cv[-1]:.0f} s)")
        with Medidor() as med: m_ = RandomForestClassifier(**hp).fit(X_aj, y_aj)
        pv, tv = prever_prefixos(m_, X_va, TAM_TODOS); por_tam = {}
        for k in TAM_GRADE:
            sub = _sub_floresta(m_, 0, k); buf = io.BytesIO(); joblib.dump(sub, buf, compress=COMPRESSAO_MODELO)
            por_tam[str(k)] = {"nos": int(sum(e.tree_.node_count for e in sub.estimators_)), "prof_media": float(np.mean([e.tree_.max_depth for e in sub.estimators_])),
                               "prof_max": int(max(e.tree_.max_depth for e in sub.estimators_)), "t_prev_val_s": float(tv[k]), "modelo_mib": buf.tell() / 2 ** 20}
        meta = {"max_depth": d, "min_samples_leaf": l, "semente": s, "t_fit_dobras_s": t_cv, "t_fit_completo_s": med.relogio, "cpu_fit_completo_s": med.cpu, "pico_ram_mib": med.pico_mib,
                "acrescimo_ram_mib": med.delta_mib, "por_tamanho": por_tam, "tam_curva": list(TAM_CURVA_USADOS), "tam_grade": TAM_GRADE}
        curva_val = np.array([auc_ovr(y_va, pv[k]) for k in TAM_CURVA_USADOS])
        tmp = arq.with_name(arq.stem + ".tmp.npz")
        np.savez_compressed(tmp, oof=oof, val=np.stack([pv[k] for k in TAM_GRADE]), curva_cv=curva_cv, curva_val=curva_val, meta=json.dumps(_limpar(meta))); os.replace(tmp, arq)
        del m_, sub, pv, oof; gc.collect(); barra.avancar(1, f"{rotulo} | treino completo ({med.relogio:.0f} s) | unidade salva"); novas += 1
        if PARAR_APOS_UNIDADES and novas >= PARAR_APOS_UNIDADES: raise RuntimeError("parada de teste (PARAR_APOS_UNIDADES)")
except KeyboardInterrupt:
    print(f"\nInterrompido. As unidades prontas ficam salvas ({sum((PASTA / 'unidades' / f'{nome_unidade(*u)}.npz').exists() for u in UNIDADES)}/{len(UNIDADES)}): rode esta célula de novo para retomar.")
    raise
else:
    interrompida = False
finally:                                                         # grava o tempo desta sessão mesmo se a busca for interrompida ou falhar
    rec = {"inicio": inicio_sessao, "fim": datetime.now().isoformat(timespec="seconds"), "relogio_s": time.perf_counter() - t_sessao, "unidades_novas": novas, "florestas_novas": novas * (K_CV + 1), "interrompida": interrompida}
    if not MODO_LEITURA: pd.DataFrame([rec]).to_csv(PASTA / "tempos_busca.csv", mode="a", header=not (PASTA / "tempos_busca.csv").exists(), index=False)
barra.fechar()
print(f"✔ Busca concluída: {len(UNIDADES)} unidades em {PASTA / 'unidades'} | tempo desta sessão: {fmt_tempo(rec['relogio_s'])}")

✔ Atalho das árvores conferido (scikit-learn 1.9.1): as primeiras 20 de 40 árvores = floresta treinada com 20
Memória livre agora: 0.5 GiB | a maior floresta (profundidade livre, 500 árvores) deve precisar de ~2.9 GiB no pico (estimativa pelo tamanho do modelo do 05m)
⚠ pode faltar memória. Feche outros programas, ou reduza o maior n_estimators (por exemplo GRADE = {'n_estimators': (100, 200, 300), ...}) antes de rodar.
Unidades prontas: 27/27  (retomando) | florestas a treinar agora: 0


Busca de hiperparâmetros: concluído em 0s
✔ Busca concluída: 27 unidades em C:\Users\Bruno\Downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_multiclasse_busca\rfm_busca_20261102_001711\unidades | tempo desta sessão: 0s


## 4. Métricas por configuração
Calcula, para cada `n_estimators` de cada unidade: as métricas multiclasse **em cada dobra** (fora da dobra), na **validação** (sem pesos e na proporção do treino) e o **recall de cada classe**. Depois junta as sementes: a **média das dobras** é o valor da validação cruzada, e o **erro-padrão** é o desvio entre dobras dividido por √5. Os custos (nós, tamanho, tempo) vêm de cada floresta.

In [11]:
# ---------- 4.1 Métricas por configuração ----------
faltam_u = [nome_unidade(*u) for u in UNIDADES if not (PASTA / "unidades" / f"{nome_unidade(*u)}.npz").exists()]
assert not faltam_u, f"faltam {len(faltam_u)} unidades: rode a célula 3.1 de novo (ela retoma de onde parou)"
M_BASE = ("roc_auc", "acuracia_balanceada", "acuracia", "precisao_macro", "f1_macro", "f1_ponderado", "mcc")
M_PROP = ("acuracia", "precisao_macro", "f1_macro", "f1_ponderado", "mcc")
linhas_fold, linhas_val, linhas_cls, linhas_curva, meta_u = [], [], [], [], {}
barra = Barra(len(UNIDADES), "Calculando as métricas")
for (d, l, s) in UNIDADES:
    z = np.load(PASTA / "unidades" / f"{nome_unidade(d, l, s)}.npz"); oof, val, meta = z["oof"], z["val"], json.loads(str(z["meta"])); meta_u[(d, l, s)] = meta
    assert not np.isnan(oof).any(), f"previsões fora da dobra incompletas em {nome_unidade(d, l, s)}"
    for i, k in enumerate(TAM_GRADE):
        base = {"n_estimators": k, "profundidade": rot(d), "min_samples_leaf": l, "semente": s}
        for f, (_, ite) in enumerate(DOBRAS, 1):
            a, _ = met(y_aj[ite], oof[i, ite]); linhas_fold.append({**base, "dobra": f, **{m: a[m] for m in M_BASE}})
        a, _ = met(y_va, val[i]); b, _ = met(y_va, val[i], w_va)
        linhas_val.append({**base, **{m: a[m] for m in M_BASE}, **{m + "_prop": b[m] for m in M_PROP}})
        for fonte, yy, PP in (("cv_fora_da_dobra", y_aj, oof[i]), ("validacao", y_va, val[i])):
            for c, r in zip(CLASSES, recall_por_classe(yy, PP)): linhas_cls.append({**base, "fonte": fonte, "classe": c, "recall": r})
    for j, k in enumerate(meta["tam_curva"]):
        linhas_curva += [{"profundidade": rot(d), "min_samples_leaf": l, "semente": s, "n_arvores": k, "fonte": "cv", "roc_auc": float(z["curva_cv"][:, j].mean())},
                         {"profundidade": rot(d), "min_samples_leaf": l, "semente": s, "n_arvores": k, "fonte": "validacao", "roc_auc": float(z["curva_val"][j])}]
    barra.avancar(1)
barra.fechar()
TEMPO_TREINO_SOMADO = float(sum(sum(m["t_fit_dobras_s"]) + m["t_fit_completo_s"] for m in meta_u.values()))      # soma dos tempos de treino de todas as florestas
tempos_sessoes = pd.read_csv(PASTA / "tempos_busca.csv") if (PASTA / "tempos_busca.csv").exists() else pd.DataFrame(columns=["relogio_s"])
TEMPO_BUSCA_RELOGIO = float(tempos_sessoes["relogio_s"].sum()) if len(tempos_sessoes) else float("nan")
print(f"\n⏱ Tempo da busca: {fmt_tempo(TEMPO_BUSCA_RELOGIO)} de relógio em {len(tempos_sessoes)} sessão(ões) (treino, previsões e gravações) | soma dos tempos de treino das {N_FLORESTAS} florestas: {fmt_tempo(TEMPO_TREINO_SOMADO)}")
df_fold, df_val, df_cls, df_curva = (pd.DataFrame(x) for x in (linhas_fold, linhas_val, linhas_cls, linhas_curva))
CHAVE = ["n_estimators", "profundidade", "min_samples_leaf"]

def resumo_cv(metrica):
    # média das sementes em cada dobra -> média e erro-padrão entre as dobras; desvio entre sementes (da média das dobras)
    por_dobra = df_fold.groupby(CHAVE + ["dobra"])[metrica].mean().reset_index(); g = por_dobra.groupby(CHAVE)[metrica]
    por_sem = df_fold.groupby(CHAVE + ["semente"])[metrica].mean().reset_index().groupby(CHAVE)[metrica].std(ddof=1)
    return pd.DataFrame({f"cv_{metrica}": g.mean(), f"cv_{metrica}_ep": g.std(ddof=1) / np.sqrt(len(DOBRAS)), f"cv_{metrica}_dp_semente": por_sem})
M_CV = ["roc_auc", "acuracia_balanceada", "acuracia", "f1_macro", "mcc"]
df_cfg = pd.concat([resumo_cv(m) for m in M_CV], axis=1)
M_VAL = ["roc_auc", "acuracia_balanceada", "acuracia", "f1_macro", "mcc", "f1_macro_prop", "mcc_prop"]
gv = df_val.groupby(CHAVE)[M_VAL]; df_cfg = df_cfg.join(gv.mean().add_prefix("val_")).join(gv.std(ddof=1).add_prefix("val_dp_"))
custo_cfg = []
for (n, d, l) in CONFIGS:
    ms = [meta_u[(d, l, s)] for s in SEMENTES]; pt = [m["por_tamanho"][str(n)] for m in ms]
    custo_cfg.append({"n_estimators": n, "profundidade": rot(d), "min_samples_leaf": l, "nos": float(np.mean([p["nos"] for p in pt])), "prof_media": float(np.mean([p["prof_media"] for p in pt])),
                      "prof_max": int(max(p["prof_max"] for p in pt)), "modelo_mib": float(np.mean([p["modelo_mib"] for p in pt])), "t_prev_val_s": float(np.mean([p["t_prev_val_s"] for p in pt])),
                      "t_treino_s": float(np.mean([m["t_fit_completo_s"] for m in ms])) * n / NMAX, "pico_ram_mib": float(np.max([m["pico_ram_mib"] for m in ms]))})
df_cfg = df_cfg.reset_index().merge(pd.DataFrame(custo_cfg), on=CHAVE); df_cfg.insert(0, "config", [f"{n}|{p}|{l}" for n, p, l in zip(df_cfg.n_estimators, df_cfg.profundidade, df_cfg.min_samples_leaf)])
print(f"✔ {len(df_cfg)} configurações | {len(df_fold):,} medidas por dobra | {len(df_val)} medidas na validação")
cols_ver = ["n_estimators", "profundidade", "min_samples_leaf", "cv_roc_auc", "cv_roc_auc_ep", "cv_mcc", "cv_mcc_ep", "cv_f1_macro", "cv_acuracia_balanceada", "val_roc_auc", "val_mcc", "val_f1_macro", "nos", "t_treino_s"]
print("\n== Todas as configurações (CV: média das dobras e sementes; ep = erro-padrão entre dobras; val = validação separada, média das sementes) ==")
display(df_cfg[cols_ver].round(5).sort_values(["n_estimators", "profundidade", "min_samples_leaf"]).reset_index(drop=True))

Calculando as métricas: concluído em 3min 12s

⏱ Tempo da busca: 5h 49min de relógio em 2 sessão(ões) (treino, previsões e gravações) | soma dos tempos de treino das 162 florestas: 5h 35min
✔ 27 configurações | 405 medidas por dobra | 81 medidas na validação

== Todas as configurações (CV: média das dobras e sementes; ep = erro-padrão entre dobras; val = validação separada, média das sementes) ==


,n_estimators,profundidade,min_samples_leaf,cv_roc_auc,cv_roc_auc_ep,cv_mcc,cv_mcc_ep,cv_f1_macro,cv_acuracia_balanceada,val_roc_auc,val_mcc,val_f1_macro,nos,t_treino_s
0,100,15,1,0.99473,0.00006,0.91204,0.00073,0.91518,0.92037,0.99477,0.91112,0.91443,4.055740e+05,26.85791
1,100,15,5,0.99463,0.00006,0.91055,0.00052,0.91362,0.91896,0.99466,0.90945,0.91256,2.639133e+05,24.24104
2,100,15,10,0.99451,0.00007,0.90893,0.00066,0.91176,0.91714,0.99457,0.90861,0.91160,2.095573e+05,26.89916
3,100,8,1,0.99225,0.00008,0.89267,0.00067,0.89455,0.90079,0.99238,0.89341,0.89564,3.122067e+04,19.00815
4,100,8,5,0.99219,0.00010,0.89237,0.00037,0.89437,0.90131,0.99239,0.89457,0.89702,2.904067e+04,18.84506
5,100,8,10,0.99218,0.00007,0.89152,0.00043,0.89354,0.90087,0.99236,0.89380,0.89596,2.778267e+04,19.07425
6,100,None,1,0.99047,0.00018,0.89874,0.00094,0.90231,0.90407,0.99015,0.89674,0.90047,3.480210e+06,32.91397
7,100,None,5,0.99489,0.00006,0.91411,0.00081,0.91773,0.92297,0.99495,0.91355,0.91732,1.026063e+06,29.55266
8,100,None,10,0.99485,0.00007,0.91193,0.00071,0.91516,0.92054,0.99492,0.91201,0.91537,5.749407e+05,28.93909
9,300,15,1,0.99476,0.00006,0.91207,0.00066,0.91522,0.92047,0.99480,0.91116,0.91446,1.213675e+06,80.57372


## 5. Escolha da combinação
Aplica a regra definida na introdução: (1) acha a **melhor** combinação em cada métrica de decisão (ROC-AUC um-contra-resto e MCC); (2) toda combinação a menos de **1 erro-padrão** da melhor **empata** com ela; (3) fica elegível quem empata **nas duas**; (4) entre as elegíveis, vence a **mais barata** (menos nós). Se nenhuma for elegível, vale a melhor posição média (plano B, avisado na tela). A tabela final compara a escolhida com o **padrão do 05m** e com as melhores de cada métrica; a posição na **validação separada** só confere, não decide.

In [12]:
# ---------- 5.1 Escolha ----------
def escolher(df, metricas_dec=METRICAS_DECISAO, mult=MULT_ERRO_PADRAO):
    d = df.reset_index(drop=True).copy(); elegivel = np.ones(len(d), bool); info = {}
    for m in metricas_dec:
        i = int(d[f"cv_{m}"].idxmax()); lim = d.loc[i, f"cv_{m}"] - mult * d.loc[i, f"cv_{m}_ep"]
        d[f"empata_{m}"] = d[f"cv_{m}"] >= lim; elegivel &= d[f"empata_{m}"].to_numpy()
        info[m] = {"melhor": d.loc[i, "config"], "valor_melhor": float(d.loc[i, f"cv_{m}"]), "erro_padrao_melhor": float(d.loc[i, f"cv_{m}_ep"]), "limite": float(lim), "n_empatadas": int(d[f"empata_{m}"].sum())}
    fallback = not elegivel.any()
    if fallback:
        rk = sum(d[f"cv_{m}"].rank(ascending=False) for m in metricas_dec); elegivel = (rk == rk.min()).to_numpy()
    cand = d[elegivel].sort_values(["nos", "n_estimators", "min_samples_leaf"], ascending=[True, True, False]).reset_index(drop=True)
    return cand.iloc[0], cand, info, fallback

ESC, CAND, INFO_SEL, FALLBACK = escolher(df_cfg)
PROF_REAL = {rot(d): d for d in GRADE["max_depth"]}
N_ESC, D_ESC, L_ESC = int(ESC["n_estimators"]), PROF_REAL[ESC["profundidade"]], int(ESC["min_samples_leaf"])
HIPERPARAMETROS_FINAL = dict(n_estimators=N_ESC, max_depth=D_ESC, min_samples_split=2, min_samples_leaf=L_ESC, max_features="sqrt", class_weight="balanced", random_state=SEMENTE, n_jobs=-1)

print("== Passo 1: melhor configuração em cada métrica de decisão e o limite de empate (melhor − 1 erro-padrão) ==")
display(pd.DataFrame(INFO_SEL).T.rename(columns={"melhor": "melhor configuração (n|prof|folha)", "valor_melhor": "valor", "erro_padrao_melhor": "erro-padrão", "limite": "limite de empate", "n_empatadas": "configs que empatam"}))
print(f"\n== Passo 2: elegíveis (empatam nas duas métricas): {len(CAND)} de {len(df_cfg)} ==" + ("   ⚠ NENHUMA empatou nas duas: usei a melhor posição média (plano B)" if FALLBACK else ""))
cols = ["config", "cv_roc_auc", "cv_roc_auc_ep", "cv_mcc", "cv_mcc_ep", "cv_f1_macro", "val_roc_auc", "val_mcc", "val_f1_macro", "nos", "modelo_mib", "t_treino_s"]
display(CAND[cols].round(5).head(15))
print(f"\n✔ ESCOLHIDA: n_estimators={N_ESC}, max_depth={D_ESC}, min_samples_leaf={L_ESC}  (a mais barata entre as {len(CAND)} elegíveis: {ESC['nos']:,.0f} nós)")

def _linha(rotulo, r): return {"configuração": rotulo, "config (n|prof|folha)": r["config"], **{c: r[c] for c in ("cv_roc_auc", "cv_roc_auc_ep", "cv_mcc", "cv_mcc_ep", "cv_f1_macro", "cv_acuracia_balanceada", "val_roc_auc", "val_mcc", "val_f1_macro", "nos", "modelo_mib", "t_treino_s", "t_prev_val_s")}}
comp = [_linha("escolhida", ESC)]
pd_ = df_cfg[(df_cfg.n_estimators == PADRAO["n_estimators"]) & (df_cfg.profundidade == rot(PADRAO["max_depth"])) & (df_cfg.min_samples_leaf == PADRAO["min_samples_leaf"])]
if len(pd_): comp.append(_linha("padrão do notebook 05m", pd_.iloc[0]))
for m in METRICAS_DECISAO: comp.append(_linha(f"melhor em {m} (CV)", df_cfg.loc[df_cfg[f"cv_{m}"].idxmax()]))
print("\n== A escolhida × o padrão × as melhores de cada métrica ==")
tab_comp = pd.DataFrame(comp); display(tab_comp.round(5))
rk_val = {m: int((df_cfg[f"val_{m}"] > ESC[f"val_{m}"]).sum()) + 1 for m in ("roc_auc", "mcc")}
print(f"Conferência na validação separada (não decide nada): a escolhida ficou em {rk_val['roc_auc']}º de {len(df_cfg)} em ROC-AUC e em {rk_val['mcc']}º em MCC.")

== Passo 1: melhor configuração em cada métrica de decisão e o limite de empate (melhor − 1 erro-padrão) ==


,melhor configuração (n|prof|folha),valor,erro-padrão,limite de empate,configs que empatam
roc_auc,500|None|5,0.994939,0.000071,0.994867,5
mcc,500|None|5,0.914239,0.000821,0.913419,3



== Passo 2: elegíveis (empatam nas duas métricas): 3 de 27 ==


,config,cv_roc_auc,cv_roc_auc_ep,cv_mcc,cv_mcc_ep,cv_f1_macro,val_roc_auc,val_mcc,val_f1_macro,nos,modelo_mib,t_treino_s
0,100|None|5,0.99489,0.00006,0.91411,0.00081,0.91773,0.99495,0.91355,0.91732,1.026063e+06,22.66577,29.55266
1,300|None|5,0.99493,0.00007,0.91419,0.00086,0.91778,0.99499,0.91333,0.91707,3.070264e+06,67.63890,88.65798
2,500|None|5,0.99494,0.00007,0.91424,0.00082,0.91782,0.99500,0.91353,0.91726,5.107577e+06,112.48931,147.76329



✔ ESCOLHIDA: n_estimators=100, max_depth=None, min_samples_leaf=5  (a mais barata entre as 3 elegíveis: 1,026,063 nós)

== A escolhida × o padrão × as melhores de cada métrica ==


,configuração,config (n|prof|folha),cv_roc_auc,cv_roc_auc_ep,cv_mcc,cv_mcc_ep,cv_f1_macro,cv_acuracia_balanceada,val_roc_auc,val_mcc,val_f1_macro,nos,modelo_mib,t_treino_s,t_prev_val_s
0,escolhida,100|None|5,0.99489,0.00006,0.91411,0.00081,0.91773,0.92297,0.99495,0.91355,0.91732,1.026063e+06,22.66577,29.55266,0.67973
1,padrão do notebook 05m,100|None|1,0.99047,0.00018,0.89874,0.00094,0.90231,0.90407,0.99015,0.89674,0.90047,3.480210e+06,54.58496,32.91397,0.78708
2,melhor em roc_auc (CV),500|None|5,0.99494,0.00007,0.91424,0.00082,0.91782,0.92296,0.99500,0.91353,0.91726,5.107577e+06,112.48931,147.76329,2.54554
3,melhor em mcc (CV),500|None|5,0.99494,0.00007,0.91424,0.00082,0.91782,0.92296,0.99500,0.91353,0.91726,5.107577e+06,112.48931,147.76329,2.54554


Conferência na validação separada (não decide nada): a escolhida ficou em 4º de 27 em ROC-AUC e em 1º em MCC.


## 6. Quanto cada hiperparâmetro influencia
- **Efeito principal:** a média de cada métrica em cada valor de um hiperparâmetro (média sobre os outros dois).
- **Tamanho do efeito:** a amplitude (maior média − menor média) comparada com o ruído (o erro-padrão): se for menor que 2× o ruído, o hiperparâmetro **não faz diferença mensurável** nessa métrica.
- **Um de cada vez, a partir do padrão** (100 árvores, sem limite de profundidade, folha 1).
- **Decomposição da variação:** quanto da diferença entre as 27 configurações vem de cada hiperparâmetro e de suas interações.
- **Gráficos:** efeitos principais, interação profundidade × folha e a curva ROC-AUC × número de árvores.

In [13]:
# ---------- 6.1 Influência dos hiperparâmetros ----------
INFL = [("ROC-AUC (CV)", "cv_roc_auc"), ("MCC (CV)", "cv_mcc"), ("F1 macro (CV)", "cv_f1_macro"), ("acurácia balanceada (CV)", "cv_acuracia_balanceada"),
        ("ROC-AUC (validação)", "val_roc_auc"), ("MCC (validação)", "val_mcc"), ("F1 macro (validação)", "val_f1_macro"),
        ("nós totais", "nos"), ("tamanho do modelo (MiB)", "modelo_mib"), ("tempo de treino (s)", "t_treino_s"), ("tempo de previsão na validação (s)", "t_prev_val_s")]
NIVEIS = {"n_estimators": TAM_GRADE, "profundidade": [rot(d) for d in GRADE["max_depth"]], "min_samples_leaf": list(GRADE["min_samples_leaf"])}
NOME_HP = {"n_estimators": "n_estimators", "profundidade": "max_depth", "min_samples_leaf": "min_samples_leaf"}
def ruido(col):
    if col.startswith("cv_") and f"{col}_ep" in df_cfg: return float(df_cfg[f"{col}_ep"].mean())
    if col.startswith("val_") and f"val_dp_{col[4:]}" in df_cfg: return float(df_cfg[f"val_dp_{col[4:]}"].mean() / np.sqrt(len(SEMENTES)))
    return float("nan")

linhas_ef, linhas_amp = [], []
for fator, niveis in NIVEIS.items():
    for nv in niveis:
        sub = df_cfg[df_cfg[fator] == nv]; linhas_ef.append({"hiperparametro": NOME_HP[fator], "valor": nv, **{r: sub[c].mean() for r, c in INFL}})
tab_ef = pd.DataFrame(linhas_ef)
for fator in NIVEIS:
    sub = tab_ef[tab_ef.hiperparametro == NOME_HP[fator]]
    for r, c in INFL:
        amp = float(sub[r].max() - sub[r].min()); rz = ruido(c)
        leitura = "efeito no custo" if rz != rz else ("ACIMA do ruído" if amp > 2 * rz else "dentro do ruído")
        linhas_amp.append({"hiperparametro": NOME_HP[fator], "métrica": r, "amplitude": amp, "ruído (erro-padrão)": rz, "amplitude / ruído": amp / rz if rz == rz and rz > 0 else float("nan"), "leitura": leitura})
tab_amp = pd.DataFrame(linhas_amp)
print("== Efeito principal: média da métrica em cada valor do hiperparâmetro (média sobre as combinações dos outros dois) ==")
display(tab_ef.round(5))
print("\n== Tamanho do efeito: amplitude (maior média − menor média) comparada com o ruído ==")
display(tab_amp.pivot(index="métrica", columns="hiperparametro", values="amplitude").loc[[r for r, _ in INFL]][[NOME_HP[f] for f in NIVEIS]].round(5))
display(tab_amp[tab_amp["métrica"].isin(["ROC-AUC (CV)", "MCC (CV)"])].round(4))

# um de cada vez, a partir do padrão
base = df_cfg[(df_cfg.n_estimators == PADRAO["n_estimators"]) & (df_cfg.profundidade == rot(PADRAO["max_depth"])) & (df_cfg.min_samples_leaf == PADRAO["min_samples_leaf"])]
tab_um = pd.DataFrame()
if len(base):
    b0 = base.iloc[0]; linhas_um = []
    for _, r in df_cfg.iterrows():
        difs = [f for f in NIVEIS if r[f] != b0[f]]
        if len(difs) == 1:
            f = difs[0]; linhas_um.append({"muda": NOME_HP[f], "de": b0[f], "para": r[f], **{f"Δ {n}": r[c] - b0[c] for n, c in INFL}, "Δ ROC-AUC (CV) em erros-padrão": (r["cv_roc_auc"] - b0["cv_roc_auc"]) / b0["cv_roc_auc_ep"],
                              "Δ MCC (CV) em erros-padrão": (r["cv_mcc"] - b0["cv_mcc"]) / b0["cv_mcc_ep"]})
    tab_um = pd.DataFrame(linhas_um); print("\n== Um de cada vez, a partir do padrão (100 árvores, sem limite de profundidade, folha 1) ==")
    display(tab_um[["muda", "de", "para", "Δ ROC-AUC (CV)", "Δ MCC (CV)", "Δ ROC-AUC (CV) em erros-padrão", "Δ MCC (CV) em erros-padrão", "Δ nós totais", "Δ tempo de treino (s)"]].round(5))
else: print("\n(o padrão do notebook 05m não está na grade: tabela 'um de cada vez' omitida)")

# decomposição da variação (três fatores, uma medida por célula)
def decompor(Y):
    a, b, c = Y.shape; mu = Y.mean(); ma, mb, mc = Y.mean((1, 2)), Y.mean((0, 2)), Y.mean((0, 1)); mab, mac, mbc = Y.mean(2), Y.mean(1), Y.mean(0)
    ss = {"n_estimators": b * c * ((ma - mu) ** 2).sum(), "max_depth": a * c * ((mb - mu) ** 2).sum(), "min_samples_leaf": a * b * ((mc - mu) ** 2).sum(),
          "n_estimators × max_depth": c * ((mab - ma[:, None] - mb[None, :] + mu) ** 2).sum(), "n_estimators × min_samples_leaf": b * ((mac - ma[:, None] - mc[None, :] + mu) ** 2).sum(),
          "max_depth × min_samples_leaf": a * ((mbc - mb[:, None] - mc[None, :] + mu) ** 2).sum()}
    res = Y - mab[:, :, None] - mac[:, None, :] - mbc[None, :, :] + ma[:, None, None] + mb[None, :, None] + mc[None, None, :] - mu
    ss["interação tripla (resíduo)"] = (res ** 2).sum(); tot = ((Y - mu) ** 2).sum()
    return {k: (float(v / tot) if tot > 0 else float("nan")) for k, v in ss.items()}
def cubo(col):
    Y = np.zeros((len(NIVEIS["n_estimators"]), len(NIVEIS["profundidade"]), len(NIVEIS["min_samples_leaf"])))
    for i, n in enumerate(NIVEIS["n_estimators"]):
        for j, p in enumerate(NIVEIS["profundidade"]):
            for k, l in enumerate(NIVEIS["min_samples_leaf"]): Y[i, j, k] = float(df_cfg[(df_cfg.n_estimators == n) & (df_cfg.profundidade == p) & (df_cfg.min_samples_leaf == l)][col].iloc[0])
    return Y
decomp = pd.DataFrame({r: decompor(cubo(c)) for r, c in INFL}); decomp.index.name = "fonte da variação"
print("\n== Decomposição da variação entre as configurações (fração de cada fonte; cada coluna soma 1) ==")
display((decomp[["ROC-AUC (CV)", "MCC (CV)", "ROC-AUC (validação)", "nós totais", "tempo de treino (s)"]] * 100).round(1).add_suffix(" (%)"))

# leitura automática
print("\n== Leitura automática (ROC-AUC e MCC da validação cruzada) ==")
for fator in NIVEIS:
    for r in ("ROC-AUC (CV)", "MCC (CV)"):
        t = tab_amp[(tab_amp.hiperparametro == NOME_HP[fator]) & (tab_amp["métrica"] == r)].iloc[0]; sub = tab_ef[tab_ef.hiperparametro == NOME_HP[fator]]
        melhor = sub.loc[sub[r].idxmax(), "valor"]
        print(f"  {NOME_HP[fator]:17s} {r:25s} amplitude {t['amplitude']:.5f} ({t['amplitude / ruído']:.1f}× o ruído) → {t['leitura']}; melhor valor na média: {melhor}")

# gráficos
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(2, 3, figsize=(12, 6.2))
    for j, fator in enumerate(NIVEIS):
        sub = tab_ef[tab_ef.hiperparametro == NOME_HP[fator]]
        for i, (r, c) in enumerate((("ROC-AUC (CV)", "cv_roc_auc"), ("MCC (CV)", "cv_mcc"))):
            ax[i, j].errorbar(range(len(sub)), sub[r], yerr=ruido(c), marker="o", capsize=4); ax[i, j].set_xticks(range(len(sub))); ax[i, j].set_xticklabels([str(v) for v in sub["valor"]]); ax[i, j].grid(alpha=.3)
            ax[i, j].set_title(f"{NOME_HP[fator]}" if i == 0 else ""); ax[i, j].set_ylabel(r if j == 0 else ""); ax[i, j].set_xlabel(NOME_HP[fator] if i == 1 else "")
    fig.suptitle("Efeito principal de cada hiperparâmetro (média sobre os outros; barra = 1 erro-padrão típico)"); fig.tight_layout(); fig.savefig(PASTA / "influencia_efeitos_principais.png", dpi=130); plt.close(fig)
    df_cfg["nos_por_arvore"] = df_cfg["nos"] / df_cfg["n_estimators"]; fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    for a_, (r, c) in zip(ax, (("ROC-AUC (CV)", "cv_roc_auc"), ("MCC (CV)", "cv_mcc"), ("nós por árvore", "nos_por_arvore"))):
        pv = df_cfg.pivot_table(index="profundidade", columns="min_samples_leaf", values=c, aggfunc="mean").reindex(index=NIVEIS["profundidade"], columns=NIVEIS["min_samples_leaf"])
        im = a_.imshow(pv.values, aspect="auto", cmap="viridis"); a_.set_xticks(range(pv.shape[1])); a_.set_xticklabels(pv.columns); a_.set_yticks(range(pv.shape[0])); a_.set_yticklabels(pv.index)
        a_.set_xlabel("min_samples_leaf"); a_.set_ylabel("max_depth"); a_.set_title(r); plt.colorbar(im, ax=a_)
        for (ii, jj), v in np.ndenumerate(pv.values): a_.text(jj, ii, f"{v:.4g}", ha="center", va="center", color="w", fontsize=8)
    fig.suptitle("Interação max_depth × min_samples_leaf (média sobre n_estimators)"); fig.tight_layout(); fig.savefig(PASTA / "influencia_interacao.png", dpi=130); plt.close(fig)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for a_, fonte, tit in zip(ax, ("cv", "validacao"), ("validação cruzada", "validação separada")):
        for (d, l) in PARES:
            t = df_curva[(df_curva.fonte == fonte) & (df_curva.profundidade == rot(d)) & (df_curva.min_samples_leaf == l)].groupby("n_arvores")["roc_auc"].mean(); a_.plot(t.index, t.values, marker="o", ms=3, label=f"prof={rot(d)} folha={l}")
        a_.set_xlabel("número de árvores"); a_.set_ylabel("ROC-AUC"); a_.set_title(f"ROC-AUC × árvores ({tit})"); a_.grid(alpha=.3)
    ax[1].legend(fontsize=7, ncol=2); fig.tight_layout(); fig.savefig(PASTA / "convergencia_arvores.png", dpi=130); plt.close(fig); print("\n✔ gráficos salvos: influencia_efeitos_principais.png, influencia_interacao.png, convergencia_arvores.png")
except ImportError:
    print("\n(matplotlib não está instalado: gráficos não gerados)")

== Efeito principal: média da métrica em cada valor do hiperparâmetro (média sobre as combinações dos outros dois) ==


,hiperparametro,valor,ROC-AUC (CV),MCC (CV),F1 macro (CV),acurácia balanceada (CV),ROC-AUC (validação),MCC (validação),F1 macro (validação),nós totais,tamanho do modelo (MiB),tempo de treino (s),tempo de previsão na validação (s)
0,n_estimators,100,0.99341,0.90365,0.90647,0.91189,0.99346,0.90370,0.90671,6.720336e+05,13.36601,25.14792,0.52979
1,n_estimators,300,0.99359,0.90398,0.90681,0.91204,0.99364,0.90418,0.90724,2.013815e+06,39.88610,75.44376,1.39635
2,n_estimators,500,0.99365,0.90404,0.90687,0.91207,0.99369,0.90426,0.90731,3.356119e+06,66.43234,125.73961,2.24119
3,max_depth,None,0.99374,0.90828,0.91176,0.91584,0.99369,0.90746,0.91111,5.074160e+06,91.94579,91.40572,1.83193
4,max_depth,15,0.99465,0.91054,0.91355,0.91887,0.99468,0.90975,0.91289,8.798139e+05,23.89332,77.99812,1.28503
5,max_depth,8,0.99226,0.89285,0.89484,0.90129,0.99242,0.89493,0.89726,8.799348e+04,3.84533,56.92746,1.05037
6,min_samples_leaf,1,0.99283,0.90129,0.90419,0.90856,0.99279,0.90078,0.90395,3.911639e+06,65.46171,78.78002,1.66330
7,min_samples_leaf,5,0.99394,0.90586,0.90875,0.91444,0.99403,0.90611,0.90920,1.316954e+06,31.40924,72.63877,1.24728
8,min_samples_leaf,10,0.99388,0.90452,0.90720,0.91299,0.99397,0.90525,0.90811,8.133753e+05,22.81350,74.91250,1.25675



== Tamanho do efeito: amplitude (maior média − menor média) comparada com o ruído ==


hiperparametro,n_estimators,max_depth,min_samples_leaf
métrica,,,
ROC-AUC (CV),2.400000e-04,2.390000e-03,1.110000e-03
MCC (CV),3.900000e-04,1.769000e-02,4.570000e-03
F1 macro (CV),4.000000e-04,1.872000e-02,4.550000e-03
acurácia balanceada (CV),1.800000e-04,1.758000e-02,5.880000e-03
ROC-AUC (validação),2.300000e-04,2.270000e-03,1.240000e-03
MCC (validação),5.600000e-04,1.482000e-02,5.330000e-03
F1 macro (validação),6.100000e-04,1.564000e-02,5.250000e-03
nós totais,2.684086e+06,4.986167e+06,3.098263e+06
tamanho do modelo (MiB),5.306634e+01,8.810046e+01,4.264822e+01


,hiperparametro,métrica,amplitude,ruído (erro-padrão),amplitude / ruído,leitura
0,n_estimators,ROC-AUC (CV),0.0002,0.0001,3.0323,ACIMA do ruído
1,n_estimators,MCC (CV),0.0004,0.0007,0.5779,dentro do ruído
11,max_depth,ROC-AUC (CV),0.0024,0.0001,30.6701,ACIMA do ruído
12,max_depth,MCC (CV),0.0177,0.0007,26.0848,ACIMA do ruído
22,min_samples_leaf,ROC-AUC (CV),0.0011,0.0001,14.2271,ACIMA do ruído
23,min_samples_leaf,MCC (CV),0.0046,0.0007,6.7366,ACIMA do ruído



== Um de cada vez, a partir do padrão (100 árvores, sem limite de profundidade, folha 1) ==


,muda,de,para,Δ ROC-AUC (CV),Δ MCC (CV),Δ ROC-AUC (CV) em erros-padrão,Δ MCC (CV) em erros-padrão,Δ nós totais,Δ tempo de treino (s)
0,max_depth,None,15,0.00426,0.01330,23.37134,14.15783,-3.074636e+06,-6.05606
1,max_depth,None,8,0.00179,-0.00607,9.80542,-6.46102,-3.448989e+06,-13.90582
2,min_samples_leaf,1,5,0.00443,0.01537,24.29878,16.36272,-2.454147e+06,-3.36131
3,min_samples_leaf,1,10,0.00438,0.01319,24.03729,14.04693,-2.905269e+06,-3.97488
4,n_estimators,100,300,0.00126,-0.00031,6.93617,-0.32774,6.945544e+06,65.82794
5,n_estimators,100,500,0.00167,-0.00025,9.15818,-0.27121,1.389719e+07,131.65587



== Decomposição da variação entre as configurações (fração de cada fonte; cada coluna soma 1) ==


,ROC-AUC (CV) (%),MCC (CV) (%),ROC-AUC (validação) (%),nós totais (%),tempo de treino (s) (%)
fonte da variação,,,,,
n_estimators,0.5,0.0,0.5,8.7,85.8
max_depth,50.7,78.7,43.0,34.8,10.2
min_samples_leaf,13.5,4.7,16.2,13.4,0.3
n_estimators × max_depth,0.6,0.1,0.6,10.3,3.0
n_estimators × min_samples_leaf,0.6,0.0,0.7,4.0,0.1
max_depth × min_samples_leaf,32.8,16.5,37.9,22.2,0.4
interação tripla (resíduo),1.2,0.0,1.2,6.6,0.1



== Leitura automática (ROC-AUC e MCC da validação cruzada) ==
  n_estimators      ROC-AUC (CV)              amplitude 0.00024 (3.0× o ruído) → ACIMA do ruído; melhor valor na média: 500
  n_estimators      MCC (CV)                  amplitude 0.00039 (0.6× o ruído) → dentro do ruído; melhor valor na média: 500
  max_depth         ROC-AUC (CV)              amplitude 0.00239 (30.7× o ruído) → ACIMA do ruído; melhor valor na média: 15
  max_depth         MCC (CV)                  amplitude 0.01769 (26.1× o ruído) → ACIMA do ruído; melhor valor na média: 15
  min_samples_leaf  ROC-AUC (CV)              amplitude 0.00111 (14.2× o ruído) → ACIMA do ruído; melhor valor na média: 5
  min_samples_leaf  MCC (CV)                  amplitude 0.00457 (6.7× o ruído) → ACIMA do ruído; melhor valor na média: 5

✔ gráficos salvos: influencia_efeitos_principais.png, influencia_interacao.png, convergencia_arvores.png


## 7. Recall por classe
O recall de cada classe na validação cruzada e na validação separada, para a combinação **escolhida** e para o **padrão** (média das sementes). O recall por classe **não decide** a escolha, mas mostra se alguma classe piorou: o ganho em ROC-AUC ou MCC pode esconder uma classe pequena (`NetBIOS`, `UDPLag`) que perdeu.

In [14]:
# ---------- 7.1 Recall por classe ----------
tab_cls_grade = df_cls.groupby(CHAVE + ["fonte", "classe"])["recall"].mean().reset_index()
def _rt(n, p, l): return tab_cls_grade[(tab_cls_grade.n_estimators == n) & (tab_cls_grade.profundidade == p) & (tab_cls_grade.min_samples_leaf == l)].pivot(index="classe", columns="fonte", values="recall").reindex(CLASSES)
rt_esc = _rt(N_ESC, ESC["profundidade"], L_ESC); out = pd.DataFrame({"n no ajuste": tab_classes["ajuste"], "n na validação": tab_classes["validação"]})
out["escolhida: CV"], out["escolhida: validação"] = rt_esc["cv_fora_da_dobra"], rt_esc["validacao"]
if len(pd_):
    rt_pad = _rt(PADRAO["n_estimators"], rot(PADRAO["max_depth"]), PADRAO["min_samples_leaf"]); out["padrão: CV"], out["padrão: validação"] = rt_pad["cv_fora_da_dobra"], rt_pad["validacao"]
    out["Δ validação (escolhida − padrão)"] = out["escolhida: validação"] - out["padrão: validação"]
print("== Recall por classe (média das sementes; a classe prevista é a de maior probabilidade) ==")
display(out.round(5))
pior_cls = rt_esc["validacao"].idxmin(); print(f"Classe com menor recall na validação (escolhida): {pior_cls} ({rt_esc.loc[pior_cls, 'validacao']:.4f}; {int(tab_classes.loc[pior_cls, 'validação'])} linhas na validação)")
if len(pd_): print(f"Maior queda de recall em relação ao padrão (validação): {out['Δ validação (escolhida − padrão)'].min():+.4f} ({out['Δ validação (escolhida − padrão)'].idxmin()}) | maior ganho: {out['Δ validação (escolhida − padrão)'].max():+.4f} ({out['Δ validação (escolhida − padrão)'].idxmax()})")

== Recall por classe (média das sementes; a classe prevista é a de maior probabilidade) ==


,n no ajuste,n na validação,escolhida: CV,escolhida: validação,padrão: CV,padrão: validação,Δ validação (escolhida − padrão)
BENIGN,40000,10000,0.99907,0.99923,0.99933,0.99907,0.00017
LDAP,24079,6020,0.97699,0.97890,0.96556,0.96440,0.01451
MSSQL,40000,10000,0.88587,0.88807,0.88314,0.87897,0.00910
NetBIOS,15592,3898,0.84894,0.84522,0.76458,0.76330,0.08192
Syn,40000,10000,0.98212,0.97883,0.95662,0.95523,0.02360
UDP,40000,10000,0.94419,0.94150,0.92408,0.91953,0.02197
UDPLag,40000,10000,0.82363,0.82500,0.83520,0.83670,-0.01170


Classe com menor recall na validação (escolhida): UDPLag (0.8250; 10000 linhas na validação)
Maior queda de recall em relação ao padrão (validação): -0.0117 (UDPLag) | maior ganho: +0.0819 (NetBIOS)


## 8. Modelo final: a combinação escolhida
Treina a combinação escolhida **no ajuste** e mede o que o notebook 05m mediu, com o **mesmo protocolo**: métricas na **validação** (sem pesos e na proporção do treino), recall e precisão por classe, **matriz de confusão**, a **dispersão entre 5 partições** da validação e a **validação cruzada** (a da busca, para a combinação escolhida, com a média das sementes em cada dobra). Em seguida mede o **custo** (seção 9) e faz o **refit** com ajuste + validação (seção 10), que é o modelo final que o notebook 08m avalia.

In [15]:
silenciar_aviso_sklearn()
HIPERPARAMETROS = dict(HIPERPARAMETROS_FINAL)   # nome usado pelas células do notebook 05m (o mesmo protocolo)
print("Combinação escolhida:", HIPERPARAMETROS)

# ---------- 8.1 Treino no ajuste ----------
if MODO_LEITURA:                                                  # busca FINALIZADA reaproveitada: carrega o modelo e os custos gravados, sem retreinar
    rf = joblib.load(PASTA / "modelo_ajuste.joblib"); _custo_gravado = json.loads((PASTA / "custo_computacional.json").read_text(encoding="utf-8"))
    class med_aj: relogio, cpu, pico_mib, delta_mib = (_custo_gravado["fit_ajuste"][k] for k in ("relogio_s", "cpu_s", "pico_ram_mib", "acrescimo_ram_mib"))
else:
    with Medidor() as med_aj:
        rf = RandomForestClassifier(**HIPERPARAMETROS).fit(X_aj, y_aj)
rf.colunas_modelo_ = list(CM)                                    # a ordem dos atributos viaja dentro do arquivo do modelo
assert list(rf.classes_) == list(range(K)), "o modelo não aprendeu todas as classes (ou a ordem delas mudou)"
fit_ajuste = {"relogio_s": med_aj.relogio, "cpu_s": med_aj.cpu, "pico_ram_mib": med_aj.pico_mib, "acrescimo_ram_mib": med_aj.delta_mib}
print(f"✔ RF {'carregado do disco (tempos da execução original)' if MODO_LEITURA else 'treinado'} no ajuste: {med_aj.relogio:.1f} s de relógio, {med_aj.cpu:.1f} s de CPU, pico de RAM {med_aj.pico_mib:,.0f} MiB (+{med_aj.delta_mib:,.0f} MiB sobre o início)")
PESO_CLASSE_AJ = {CLASSES[k]: float(len(y_aj) / (K * (y_aj == k).sum())) for k in range(K)}
print("Pesos de classe do ajuste (balanced):", {c: round(v, 3) for c, v in PESO_CLASSE_AJ.items()})

# ---------- 8.2 Métricas na validação ----------
proba_va = rf.predict_proba(X_va)
m_sem, arr_sem = metricas_multi(y_va, proba_va, None); m_prop, arr_prop = metricas_multi(y_va, proba_va, w_va)
DEPENDE = {k: "sim" for k in ("acuracia", "precisao_macro", "f1_macro", "f1_ponderado", "mcc")}; DEPENDE["roc_auc_ovr_macro"] = "pouco"
ordem = ["roc_auc_ovr_macro", "acuracia_balanceada", "acuracia", "precisao_macro", "f1_macro", "f1_ponderado", "mcc"]
tab_val = pd.DataFrame({"sem pesos": {k: m_sem[k] for k in ordem}, "proporção do treino": {k: m_prop[k] for k in ordem}}); tab_val["depende da proporção?"] = [DEPENDE.get(k, "não") for k in tab_val.index]
print("\n== Métricas na validação (previsão = classe de maior probabilidade) ==  (a coluna 'proporção do treino' usa o peso de cada linha da validação)")
display(tab_val.round(5))
tab_cls = pd.DataFrame({"suporte (validação)": arr_sem["suporte"].astype(int), "recall": arr_sem["recall"], "precisão (sem pesos)": arr_sem["precisao"], "F1 (sem pesos)": arr_sem["f1"], "precisão (prop. treino)": arr_prop["precisao"], "F1 (prop. treino)": arr_prop["f1"],
                        "ROC-AUC um-contra-resto": arr_sem["auc"], "peso de classe no ajuste": [PESO_CLASSE_AJ[c] for c in CLASSES]}, index=CLASSES)
print("\n== Por classe (recall e ROC-AUC não dependem da proporção) =="); display(tab_cls.round(4))
cm_val = pd.DataFrame(arr_sem["cm"].astype(int), index=[f"real {c}" for c in CLASSES], columns=[f"prev. {c}" for c in CLASSES])
print("\n== Matriz de confusão na validação (linhas = classe real; colunas = classe prevista) =="); display(cm_val)
cm_rec = pd.DataFrame(arr_sem["cm"] / arr_sem["cm"].sum(axis=1, keepdims=True), index=cm_val.index, columns=cm_val.columns)
print("== Mesma matriz, em fração da classe real (a diagonal é o recall) =="); display(cm_rec.round(4))
fora = [(CLASSES[i], CLASSES[j], int(arr_sem["cm"][i, j]), arr_sem["cm"][i, j] / arr_sem["cm"][i].sum()) for i in range(K) for j in range(K) if i != j and arr_sem["cm"][i, j] > 0]
fora = sorted(fora, key=lambda x: -x[2])[:8]
print("Confusões mais frequentes (real → previsto): " + " | ".join(f"{a}→{b}: {n:,} ({100 * f:.2f}% de {a})" for a, b, n, f in fora))

# ---------- 8.3 Dispersão entre partições da validação ----------
linhas_part = []
for k, (_, idx) in enumerate(StratifiedKFold(K_PARTICOES_VAL, shuffle=True, random_state=SEMENTE).split(X_va, y_va), 1):
    e1, _a = metricas_multi(y_va[idx], proba_va[idx], None); e2, _b = metricas_multi(y_va[idx], proba_va[idx], w_va[idx])
    linhas_part.append({"particao": k, "n": len(idx), "roc_auc_ovr_macro": e1["roc_auc_ovr_macro"], "acuracia_balanceada": e1["acuracia_balanceada"], "acuracia": e1["acuracia"], "f1_macro": e1["f1_macro"], "mcc": e1["mcc"], "f1_macro_prop": e2["f1_macro"], "mcc_prop": e2["mcc"]})
tab_part = pd.DataFrame(linhas_part); resumo_part = tab_part.drop(columns=["particao", "n"]).agg(["mean", "std", "min", "max"]).T
print(f"\n== Dispersão entre {K_PARTICOES_VAL} partições da validação =="); display(tab_part.round(5)); display(resumo_part.round(5))

# ---------- 8.4 Validação cruzada (da busca) para a combinação escolhida ----------
sel_cv = df_fold[(df_fold.n_estimators == N_ESC) & (df_fold.profundidade == ESC["profundidade"]) & (df_fold.min_samples_leaf == L_ESC)]
tab_cv = sel_cv.groupby("dobra")[["roc_auc", "acuracia_balanceada", "acuracia", "f1_macro", "mcc"]].mean().reset_index()
resumo_cv = tab_cv.drop(columns=["dobra"]).agg(["mean", "std", "min", "max"]).T
tab_cv_cls = rt_esc[["cv_fora_da_dobra"]].T.rename(index={"cv_fora_da_dobra": "média das dobras e sementes"}).reset_index(names="origem")
print(f"\n== Validação cruzada da combinação escolhida ({K_CV} dobras; média das {len(SEMENTES)} sementes em cada dobra) ==")
display(tab_cv.round(5)); display(resumo_cv.round(5))
print("Recall por classe na validação cruzada (média das dobras e sementes):"); display(tab_cv_cls.round(4))

Combinação escolhida: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
✔ RF carregado do disco (tempos da execução original) no ajuste: 39.3 s de relógio, 279.2 s de CPU, pico de RAM 936 MiB (+333 MiB sobre o início)
Pesos de classe do ajuste (balanced): {'BENIGN': 0.856, 'LDAP': 1.422, 'MSSQL': 0.856, 'NetBIOS': 2.196, 'Syn': 0.856, 'UDP': 0.856, 'UDPLag': 0.856}

== Métricas na validação (previsão = classe de maior probabilidade) ==  (a coluna 'proporção do treino' usa o peso de cada linha da validação)


,sem pesos,proporção do treino,depende da proporção?
roc_auc_ovr_macro,0.99492,0.99364,pouco
acuracia_balanceada,0.92246,0.92246,não
acuracia,0.92655,0.93292,sim
precisao_macro,0.91428,0.81977,sim
f1_macro,0.91741,0.85494,sim
f1_ponderado,0.92640,0.93964,sim
mcc,0.91361,0.88140,sim



== Por classe (recall e ROC-AUC não dependem da proporção) ==


,suporte (validação),recall,precisão (sem pesos),F1 (sem pesos),precisão (prop. treino),F1 (prop. treino),ROC-AUC um-contra-resto,peso de classe no ajuste
BENIGN,10000,0.9993,0.9977,0.9985,0.9961,0.9977,1.0000,0.8560
LDAP,6020,0.9784,0.9559,0.9670,0.8592,0.9149,0.9992,1.4219
MSSQL,10000,0.8884,0.9351,0.9111,0.9717,0.9282,0.9948,0.8560
NetBIOS,3898,0.8458,0.7531,0.7968,0.3987,0.5419,0.9892,2.1959
Syn,10000,0.9790,0.9447,0.9615,0.9676,0.9733,0.9987,0.8560
UDP,10000,0.9409,0.8952,0.9175,0.9904,0.9650,0.9942,0.8560
UDPLag,10000,0.8254,0.9183,0.8694,0.5547,0.6635,0.9884,0.8560



== Matriz de confusão na validação (linhas = classe real; colunas = classe prevista) ==


,prev. BENIGN,prev. LDAP,prev. MSSQL,prev. NetBIOS,prev. Syn,prev. UDP,prev. UDPLag
real BENIGN,9993,2,0,5,0,0,0
real LDAP,7,5890,60,59,1,0,3
real MSSQL,1,204,8884,908,0,0,3
real NetBIOS,6,51,532,3297,1,0,11
real Syn,5,1,0,6,9790,0,198
real UDP,0,2,22,48,0,9409,519
real UDPLag,4,12,3,55,571,1101,8254


... (6 outputs left out)

,dobra,roc_auc,acuracia_balanceada,acuracia,f1_macro,mcc
0,1,0.99477,0.92143,0.92607,0.91645,0.91312
1,2,0.99497,0.92376,0.92739,0.91801,0.91471
2,3,0.99501,0.92531,0.92896,0.92003,0.91652
3,4,0.99500,0.92385,0.92728,0.91861,0.91448
4,5,0.99472,0.92051,0.92492,0.91556,0.91170


,mean,std,min,max
roc_auc,0.99489,0.00014,0.99472,0.99501
acuracia_balanceada,0.92297,0.00196,0.92051,0.92531
acuracia,0.92692,0.00152,0.92492,0.92896
f1_macro,0.91773,0.00177,0.91556,0.92003
mcc,0.91411,0.00181,0.91170,0.91652


Recall por classe na validação cruzada (média das dobras e sementes):


classe,origem,BENIGN,LDAP,MSSQL,NetBIOS,Syn,UDP,UDPLag
0,média das dobras e sementes,0.9991,0.977,0.8859,0.8489,0.9821,0.9442,0.8236


## 9. Custo computacional do modelo do ajuste
O mesmo cálculo do notebook 05m: tamanho do arquivo, número de árvores e de nós, **latência** de 1 fluxo e **vazão** em lote (com `n_jobs=1` e com todos os núcleos, com e sem o pré-processamento).

In [16]:
silenciar_aviso_sklearn()
# ---------- 9.1 Custo do modelo da fase 1 ----------
if MODO_LEITURA: tam_aj, inf_aj = _custo_gravado["modelo_ajuste"], _custo_gravado["inferencia_modelo_ajuste"]      # custos medidos na execução original
else:
    tam_aj = tamanho_e_nos(rf, PASTA / "modelo_ajuste.joblib", COMPRESSAO_MODELO)
    inf_aj = medir_inferencia(rf, prep, X_va, X_va_raw, n_lat=N_LATENCIA, semente=SEMENTE)
print(f"Modelo do ajuste: {tam_aj['arquivo_mib']:.1f} MiB em disco | {tam_aj['n_arvores']} árvores | {tam_aj['nos_total']:,} nós | profundidade média {tam_aj['profundidade_media']:.1f} (máx {tam_aj['profundidade_max']})")
display(pd.DataFrame(inf_aj).set_index("n_jobs").T.round(3))

Modelo do ajuste: 23.0 MiB em disco | 100 árvores | 1,045,332 nós | profundidade média 46.4 (máx 78)


n_jobs,1,-1
n_latencia,1000.000,1000.000
latencia_modelo_p50_ms,27.190,67.352
latencia_modelo_p95_ms,33.102,97.880
latencia_modelo_p99_ms,40.930,120.001
latencia_fim_a_fim_p50_ms,27.593,67.596
latencia_fim_a_fim_p95_ms,33.971,98.169
latencia_fim_a_fim_p99_ms,41.961,121.254
vazao_modelo_fluxos_s,19283.409,99386.180
vazao_fim_a_fim_fluxos_s,19224.518,89567.250
n_lote,59918.000,59918.000


## 10. Refit (modelo final)
Refaz a combinação escolhida com **todas as linhas do refit** (as mesmas do notebook 05m: ajuste + validação), `REPETICOES_TEMPO` vezes, só para medir o tempo de forma confiável. O notebook confere que as repetições dão o **mesmo modelo**. O primeiro é o **modelo final**, gravado para o notebook 08m.

In [17]:
silenciar_aviso_sklearn()
# ---------- 10.1 Refit ----------
if MODO_LEITURA:
    modelo_final = joblib.load(PASTA / "modelo_final_refit.joblib"); reps = _custo_gravado["fit_refit_repeticoes"]; p_ref = modelo_final.predict_proba(X_va[:5000])
else:
    reps, modelo_final, p_ref = [], None, None
    for k in range(1, REPETICOES_TEMPO + 1):
        with Medidor() as m:
            mdl = RandomForestClassifier(**HIPERPARAMETROS).fit(X_rf, y_rf)
        reps.append({"repeticao": k, "relogio_s": m.relogio, "cpu_s": m.cpu, "pico_ram_mib": m.pico_mib, "acrescimo_ram_mib": m.delta_mib})
        if modelo_final is None:
            modelo_final = mdl; modelo_final.colunas_modelo_ = list(CM); p_ref = mdl.predict_proba(X_va[:5000])
        else:
            assert np.allclose(mdl.predict_proba(X_va[:5000]), p_ref), "as repetições do refit deram modelos diferentes (a semente não está fixando o resultado)"
        del mdl
assert list(modelo_final.classes_) == list(range(K))
tab_reps = pd.DataFrame(reps); display(tab_reps.round(3))
fit_refit = {"repeticoes": REPETICOES_TEMPO, "relogio_mediana_s": float(tab_reps.relogio_s.median()), "relogio_min_s": float(tab_reps.relogio_s.min()), "relogio_max_s": float(tab_reps.relogio_s.max()),
             "cpu_mediana_s": float(tab_reps.cpu_s.median()), "pico_ram_max_mib": float(tab_reps.pico_ram_mib.max()), "acrescimo_ram_max_mib": float(tab_reps.acrescimo_ram_mib.max())}
PESO_CLASSE_RF = {CLASSES[k]: float(len(y_rf) / (K * (y_rf == k).sum())) for k in range(K)}
if MODO_LEITURA: tam_fin, inf_fin = _custo_gravado["modelo_final"], _custo_gravado["inferencia_modelo_final"]
else:
    tam_fin = tamanho_e_nos(modelo_final, PASTA / "modelo_final_refit.joblib", COMPRESSAO_MODELO)
    inf_fin = medir_inferencia(modelo_final, prep, X_va, X_va_raw, n_lat=N_LATENCIA, semente=SEMENTE)
print(f"\n✔ Refit: {fit_refit['relogio_mediana_s']:.1f} s (mediana de {REPETICOES_TEMPO}; mín {fit_refit['relogio_min_s']:.1f}, máx {fit_refit['relogio_max_s']:.1f}), CPU {fit_refit['cpu_mediana_s']:.1f} s, pico de RAM {fit_refit['pico_ram_max_mib']:,.0f} MiB")
print(f"Pesos de classe do refit (balanced): { {c: round(v, 3) for c, v in PESO_CLASSE_RF.items()} }")
print(f"Modelo final: {tam_fin['arquivo_mib']:.1f} MiB em disco | {tam_fin['nos_total']:,} nós | profundidade média {tam_fin['profundidade_media']:.1f} (máx {tam_fin['profundidade_max']})")
print("(latência e vazão do modelo final: medidas na validação só para VELOCIDADE; não são métricas de qualidade)")
display(pd.DataFrame(inf_fin).set_index("n_jobs").T.round(3))

,repeticao,relogio_s,cpu_s,pico_ram_mib,acrescimo_ram_mib
0,1,54.409,361.375,1011.234,413.758
1,2,48.511,345.781,1156.469,388.047
2,3,47.121,334.062,1083.562,191.844



✔ Refit: 48.5 s (mediana de 3; mín 47.1, máx 54.4), CPU 345.8 s, pico de RAM 1,156 MiB
Pesos de classe do refit (balanced): {'BENIGN': 0.856, 'LDAP': 1.422, 'MSSQL': 0.856, 'NetBIOS': 2.196, 'Syn': 0.856, 'UDP': 0.856, 'UDPLag': 0.856}
Modelo final: 27.7 MiB em disco | 1,267,902 nós | profundidade média 47.5 (máx 76)
(latência e vazão do modelo final: medidas na validação só para VELOCIDADE; não são métricas de qualidade)


n_jobs,1,-1
n_latencia,1000.000,1000.000
latencia_modelo_p50_ms,18.616,104.290
latencia_modelo_p95_ms,20.897,155.815
latencia_modelo_p99_ms,25.934,230.583
latencia_fim_a_fim_p50_ms,18.809,104.623
latencia_fim_a_fim_p95_ms,21.245,156.540
latencia_fim_a_fim_p99_ms,26.093,230.887
vazao_modelo_fluxos_s,19069.035,67100.392
vazao_fim_a_fim_fluxos_s,19260.242,66172.477
n_lote,59918.000,59918.000


## 11. Gravação
Grava na pasta da busca: os resultados de toda a grade, a escolha e as métricas do modelo final, a configuração (com as classes, os pesos e a **impressão digital das dobras**), o pré-processamento, o custo e os dois modelos. Acrescenta uma linha a `registro_buscas.csv`.

In [18]:
if MODO_LEITURA:
    print("Modo leitura: nada foi regravado (os arquivos da busca finalizada ficam como estão).")
else:
    # ---------- 11.1 Gravação ----------
    hw = {"so": platform.platform(), "python": sys.version.split()[0], "cpu": platform.processor(), "nucleos_logicos": os.cpu_count()}
    if psutil: hw.update({"nucleos_fisicos": psutil.cpu_count(logical=False), "ram_total_gib": psutil.virtual_memory().total / 2 ** 30})
    sw = {"scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__, "joblib": joblib.__version__, "psutil": None if psutil is None else psutil.__version__}
    protocolo = {"k_particoes_validacao": K_PARTICOES_VAL, "k_cv_ajuste": K_CV, "repeticoes_tempo_refit": REPETICOES_TEMPO, "n_latencia": N_LATENCIA}
    config = {"run_id": RUN_ID, "data": datetime.now().isoformat(timespec="seconds"), "modelo": "RandomForestClassifier (multiclasse)", "variante": "sem_auc_ajustado", "origem": "notebook 07m (busca de hiperparâmetros)",
              "hiperparametros": HIPERPARAMETROS_FINAL, "semente": SEMENTE, "previsao": "classe de maior probabilidade (argmax)", "versao_selecao_notebook04": versao_sel, "versao_views_notebook03": versao_views,
              "classes": {"nomes": CLASSES, "codigos": COD}, "pesos_classe_balanced_ajuste": PESO_CLASSE_AJ, "pesos_classe_balanced_refit": PESO_CLASSE_RF,
              "dados": {"ajuste": len(X_aj), "validacao": len(X_va), "refit": len(X_rf), "linhas_treino_por_classe": N_TREINO, "ajuste_por_classe": N_AJ, "validacao_por_classe": N_VA, "usadas_por_classe": {c: N_AJ[c] + N_VA[c] for c in CLASSES}},
              "protocolo": protocolo, "software": sw, "hardware": hw}
    salvar_json(config, PASTA / "config_treino.json"); salvar_json(prep.para_dict(), PASTA / "preprocessamento.json"); salvar_json({"classes": CLASSES, "codigos": COD}, PASTA / "classes.json")
    _criado_em = json.loads((PASTA / "config_busca.json").read_text(encoding="utf-8")).get("criado_em")
    cfg_busca = {"run_id": RUN_ID, "impressao": IMPRESSAO, "impressao_dobras": IMPRESSAO_DOBRAS, "retomada": RETOMANDO, "criado_em": _criado_em, "concluido_em": datetime.now().isoformat(timespec="seconds"),
                 "tempo_busca": {"relogio_s": TEMPO_BUSCA_RELOGIO, "sessoes": len(tempos_sessoes), "treino_somado_s": TEMPO_TREINO_SOMADO, "florestas": N_FLORESTAS, "detalhe_sessoes": tempos_sessoes.to_dict(orient="records")},
                 "grade": GRADE, "sementes": SEMENTES, "k_cv": K_CV, "semente_dobras": SEMENTE, "tamanhos_avaliados": TAM_TODOS, "n_florestas": N_FLORESTAS, "fixos": FIXOS,
                 "regra": {"metricas": METRICAS_DECISAO, "mult_erro_padrao": MULT_ERRO_PADRAO, "desempate": "menos nós, depois menos árvores, depois folha maior"}, "classes": CLASSES, "software": sw, "hardware": hw}
    salvar_json(cfg_busca, PASTA / "config_busca.json")
    salvar_json({"escolhida": {"n_estimators": N_ESC, "max_depth": D_ESC, "min_samples_leaf": L_ESC, "config": ESC["config"]}, "usou_plano_b": bool(FALLBACK), "passo1": INFO_SEL, "n_elegiveis": len(CAND),
                 "elegiveis": CAND[["config", "cv_roc_auc", "cv_mcc", "nos"]].to_dict(orient="records"), "hiperparametros_finais": HIPERPARAMETROS_FINAL}, PASTA / "selecao_hiperparametros.json")
    df_cfg.to_csv(PASTA / "resultados_grade.csv", index=False); df_fold.to_csv(PASTA / "cv_dobras.csv", index=False); df_val.to_csv(PASTA / "validacao_por_config.csv", index=False)
    tab_cls_grade.to_csv(PASTA / "recall_por_classe_grade.csv", index=False); tab_comp.to_csv(PASTA / "comparacao_escolhida_padrao.csv", index=False); out.reset_index(names="classe").to_csv(PASTA / "recall_por_classe_escolhida_padrao.csv", index=False)
    tab_ef.to_csv(PASTA / "efeitos_principais.csv", index=False); tab_amp.to_csv(PASTA / "amplitude_efeitos.csv", index=False); decomp.reset_index().to_csv(PASTA / "decomposicao_variacao.csv", index=False)
    df_curva.to_csv(PASTA / "convergencia_arvores.csv", index=False)
    if len(tab_um): tab_um.to_csv(PASTA / "efeito_um_de_cada_vez.csv", index=False)
    met_val = {"sem_pesos": m_sem, "proporcao_treino": m_prop, "por_classe": tab_cls.to_dict(orient="index"), "matriz_confusao": cm_val.to_dict(orient="index"),
               "dispersao_particoes": {"k": K_PARTICOES_VAL, "resumo": resumo_part.to_dict(orient="index")}, "cv_ajuste": {"k": K_CV, "origem": "busca (média das sementes por dobra)", "resumo": resumo_cv.to_dict(orient="index")}}
    salvar_json(met_val, PASTA / "metricas_validacao.json")
    tab_val.reset_index(names="metrica").to_csv(PASTA / "metricas_validacao.csv", index=False); tab_cls.reset_index(names="classe").to_csv(PASTA / "metricas_por_classe_validacao.csv", index=False)
    cm_val.reset_index(names="classe_real").to_csv(PASTA / "matriz_confusao_validacao.csv", index=False); tab_part.to_csv(PASTA / "dispersao_particoes_validacao.csv", index=False)
    tab_cv.to_csv(PASTA / "cv_ajuste.csv", index=False); tab_cv_cls.to_csv(PASTA / "cv_recall_por_classe.csv", index=False)
    custo = {"fit_ajuste": fit_ajuste, "fit_refit": fit_refit, "fit_refit_repeticoes": reps, "modelo_ajuste": tam_aj, "modelo_final": tam_fin, "inferencia_modelo_ajuste": inf_aj, "inferencia_modelo_final": inf_fin}
    salvar_json(custo, PASTA / "custo_computacional.json")
    linhas_custo = [{"item": "fit_ajuste", **fit_ajuste}, {"item": "fit_refit", **fit_refit}] + [{"item": f"inferencia_final_njobs{r['n_jobs']}", **r} for r in inf_fin] + [{"item": "modelo_final", **tam_fin}]
    pd.DataFrame(linhas_custo).to_csv(PASTA / "custo_computacional.csv", index=False)
    refit_ids = rf_df[["id_linha"]].assign(y=y_rf); refit_ids.to_csv(PASTA / "refit_ids.csv", index=False)
    i1 = next(r for r in inf_fin if r["n_jobs"] == 1); iN = inf_fin[-1]
    registro = {"run_id": RUN_ID, "data": config["data"], "retomada": RETOMANDO, "n_configuracoes": len(CONFIGS), "sementes": len(SEMENTES), "k_cv": K_CV, "escolhida_n_estimators": N_ESC, "escolhida_max_depth": rot(D_ESC),
                "escolhida_min_samples_leaf": L_ESC, "tempo_busca_relogio_s": TEMPO_BUSCA_RELOGIO, "tempo_treino_somado_s": TEMPO_TREINO_SOMADO, "n_sessoes": len(tempos_sessoes), "usou_plano_b": bool(FALLBACK), "n_elegiveis": len(CAND),
                "cv_roc_auc": ESC["cv_roc_auc"], "cv_roc_auc_ep": ESC["cv_roc_auc_ep"], "cv_mcc": ESC["cv_mcc"], "cv_mcc_ep": ESC["cv_mcc_ep"], "cv_f1_macro": ESC["cv_f1_macro"],
                "roc_auc_val": m_sem["roc_auc_ovr_macro"], "acuracia_balanceada_val": m_sem["acuracia_balanceada"], "f1_macro_val": m_sem["f1_macro"], "mcc_val": m_sem["mcc"], "f1_macro_prop_val": m_prop["f1_macro"], "mcc_prop_val": m_prop["mcc"],
                "fit_refit_relogio_mediana_s": fit_refit["relogio_mediana_s"], "modelo_final_mib": tam_fin["arquivo_mib"], "nos_total": tam_fin["nos_total"],
                "latencia_p50_ms_njobs1": i1["latencia_fim_a_fim_p50_ms"], "latencia_p99_ms_njobs1": i1["latencia_fim_a_fim_p99_ms"], "vazao_fluxos_s_njobs1": i1["vazao_fim_a_fim_fluxos_s"], "vazao_fluxos_s_njobs_orig": iN["vazao_fim_a_fim_fluxos_s"], "pasta": str(PASTA)}
    caminho_reg = ARTEFATOS_BASE / "registro_buscas.csv"; pd.DataFrame([registro]).to_csv(caminho_reg, mode="a", header=not caminho_reg.exists(), index=False)
    print(f"✔ Artefatos gravados em {PASTA}\n✔ Registro acrescentado a {caminho_reg} ({sum(1 for _ in open(caminho_reg, encoding='utf-8')) - 1} busca(s) registrada(s))")
    for f in sorted(PASTA.iterdir()):
        if f.is_file(): print(f"   {f.name:42s} {f.stat().st_size / 2**10:>10,.1f} KiB")


Modo leitura: nada foi regravado (os arquivos da busca finalizada ficam como estão).


## 12. Conferências
Confere, de forma independente: que a busca está completa, que as dobras cobrem cada linha do ajuste uma vez (com todas as classes), que o **atalho das árvores** vale, que a escolhida treinada direto **reproduz a previsão da busca**, que os modelos gravados reproduzem as previsões, que as linhas do ajuste, da validação e do refit são as do notebook 04m, que a escolha é **reproduzível** a partir do CSV gravado e que as matrizes e os pesos fecham.

In [19]:
# ---------- 12.1 Conferências ----------
from sklearn.utils.class_weight import compute_class_weight
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)

refit_sel_ids = consultar("SELECT toString(id_linha) AS id_linha FROM " + TAB_REFIT)
verifica("Busca completa: todas as unidades gravadas", all((PASTA / "unidades" / f"{nome_unidade(*u)}.npz").exists() for u in UNIDADES), f"({len(UNIDADES)} unidades, {N_FLORESTAS} florestas)")
verifica("Dobras cobrem cada linha do ajuste exatamente uma vez", np.array_equal(np.sort(np.concatenate([ite for _, ite in DOBRAS])), np.arange(len(y_aj))))
verifica("Toda classe aparece em todas as dobras", all(len(np.unique(y_aj[ite])) == K for _, ite in DOBRAS), f"({K} classes)")
_d0, _l0, _s0 = next(((d, l, s) for (d, l, s) in UNIDADES if s == SEMENTE), UNIDADES[0]); _k0 = TAM_GRADE[0]
_z = np.load(PASTA / "unidades" / f"{nome_unidade(_d0, _l0, _s0)}.npz"); _dir = RandomForestClassifier(n_estimators=_k0, max_depth=_d0, min_samples_leaf=_l0, random_state=_s0, **FIXOS).fit(X_aj, y_aj)
_dif = float(np.abs(_z["val"][0] - _dir.predict_proba(X_va)).max())
verifica(f"Atalho das árvores: as primeiras {_k0} de {NMAX} = floresta treinada com {_k0} (prof={rot(_d0)}, folha={_l0}, semente={_s0})", _dif < 1e-6, f"(diferença máxima nas probabilidades: {_dif:.1e})"); del _dir
if SEMENTE in SEMENTES:
    _zz = np.load(PASTA / "unidades" / f"{nome_unidade(D_ESC, L_ESC, SEMENTE)}.npz")
    verifica("A combinação escolhida treinada direto reproduz a previsão da busca", np.allclose(_zz["val"][TAM_GRADE.index(N_ESC)], proba_va, atol=1e-6))
m_ajuste_lido = joblib.load(PASTA / "modelo_ajuste.joblib"); m_final_lido = joblib.load(PASTA / "modelo_final_refit.joblib")
verifica("Modelo do ajuste gravado reproduz as probabilidades da validação", np.allclose(m_ajuste_lido.predict_proba(X_va), proba_va))
verifica("Modelo final gravado reproduz as previsões do modelo em memória", np.allclose(m_final_lido.predict_proba(X_va[:5000]), p_ref))
prep_lido = json.loads((PASTA / "preprocessamento.json").read_text(encoding="utf-8"))
verifica("Ordem dos atributos gravada = ordem usada no treino e dentro do modelo", prep_lido["colunas_modelo"] == CM == list(m_final_lido.colunas_modelo_) == list(m_ajuste_lido.colunas_modelo_))
verifica("Medianas gravadas = medianas da tabela de auditoria do notebook 03m", prep_lido["medianas_treino"] == MEDIANAS and set(prep_lido["colunas_retiradas_ablacao"]) == set(RETIRADAS))
verifica("Hiperparâmetros do modelo final = combinação escolhida (inclui class_weight='balanced')", all(getattr(m_final_lido, k) == v for k, v in HIPERPARAMETROS_FINAL.items()))
verifica("O modelo aprendeu as classes 0..K-1, na ordem de classes.json", list(m_final_lido.classes_) == list(m_ajuste_lido.classes_) == list(range(K)))
verifica("Ajuste e validação com as linhas planejadas por classe (notebook 04m)", conta(y_aj) == N_AJ and conta(y_va) == N_VA, f"({len(X_aj):,} e {len(X_va):,})")
verifica("Refit = ajuste + validação: todas as linhas selecionadas, por classe, e as mesmas do notebook 05m", conta(y_rf) == {c: N_AJ[c] + N_VA[c] for c in CLASSES} and set(aj["id_linha"]) | set(va["id_linha"]) == set(rf_df["id_linha"]) == set(refit_sel_ids["id_linha"]), f"({len(X_rf):,} linhas)")
_esc2 = escolher(pd.read_csv(PASTA / "resultados_grade.csv"))[0]
verifica("A escolha é reproduzível a partir do resultados_grade.csv gravado", _esc2["config"] == ESC["config"], f"({ESC['config']})")
verifica("Decomposição da variação: cada coluna soma 100%", bool(np.allclose(decomp.sum(), 1.0, atol=1e-9)))
verifica("Pesos de classe (balanced) conferem com o scikit-learn", all(abs(PESO_CLASSE_AJ[CLASSES[k]] - w) < 1e-12 for k, w in enumerate(compute_class_weight("balanced", classes=np.arange(K), y=y_aj))))
verifica("Probabilidades finitas, entre 0 e 1, cada linha soma 1, e as matrizes de confusão fecham", bool(np.isfinite(proba_va).all() and ((proba_va >= 0) & (proba_va <= 1)).all() and np.allclose(proba_va.sum(axis=1), 1.0)) and int(arr_sem["cm"].sum()) == len(y_va) and abs(arr_prop["cm"].sum() - float(w_va.sum())) < 1e-6)
verifica("Pesos da validação somam as linhas de cada classe no treino", all(abs(float(w_va[y_va == COD[c]].sum()) - N_TREINO[c]) < 1e-6 * N_TREINO[c] for c in CLASSES))
verifica("Métricas que não dependem da proporção são iguais nas duas colunas", abs(m_sem["acuracia_balanceada"] - m_prop["acuracia_balanceada"]) < 1e-9 and np.allclose(arr_sem["recall"], arr_prop["recall"], atol=1e-12))
verifica("Recall por classe: toda classe tem linhas na validação", bool((tab_classes["validação"] > 0).all()))
verifica("A impressão digital das dobras foi gravada na configuração da busca", json.loads((PASTA / "config_busca.json").read_text(encoding="utf-8")).get("impressao_dobras") == IMPRESSAO_DOBRAS, f"({IMPRESSAO_DOBRAS})")
verifica("Arquivos esperados gravados", all((PASTA / f).exists() for f in ("modelo_ajuste.joblib", "modelo_final_refit.joblib", "config_treino.json", "config_busca.json", "preprocessamento.json", "classes.json", "metricas_validacao.json", "metricas_validacao.csv",
                                                                       "metricas_por_classe_validacao.csv", "matriz_confusao_validacao.csv", "dispersao_particoes_validacao.csv", "cv_ajuste.csv", "custo_computacional.json", "custo_computacional.csv", "selecao_hiperparametros.json",
                                                                       "tempos_busca.csv", "resultados_grade.csv", "cv_dobras.csv", "validacao_por_config.csv", "recall_por_classe_grade.csv", "efeitos_principais.csv", "decomposicao_variacao.csv", "convergencia_arvores.csv", "refit_ids.csv")))
verifica("Custo medido: tempos positivos" + ("" if psutil else " (memória não medida: psutil ausente)"), fit_ajuste["relogio_s"] > 0 and fit_refit["relogio_mediana_s"] > 0 and inf_fin[0]["vazao_fim_a_fim_fluxos_s"] > 0)
assert all(ok), "há checagens reprovadas acima"
if not MODO_LEITURA: (PASTA / "FINALIZADO").write_text(datetime.now().isoformat(timespec="seconds"), encoding="utf-8")
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas.")
print(f"\nPróximo passo: notebook 08m (avaliação no teste, uma vez), apontando para {PASTA.name}")

  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...
✔ Busca completa: todas as unidades gravadas (27 unidades, 162 florestas)
✔ Dobras cobrem cada linha do ajuste exatamente uma vez 
✔ Toda classe aparece em todas as dobras (7 classes)
✔ Atalho das árvores: as primeiras 100 de 500 = floresta treinada com 100 (prof=None, folha=1, semente=42) (diferença máxima nas probabilidades: 3.0e-08)
✔ A combinação escolhida treinada direto reproduz a previsão da busca 
✔ Modelo do ajuste gravado reproduz as probabilidades da validação 
✔ Modelo final gravado reproduz as previsões do modelo em memória 
✔ Ordem dos atributos gravada = ordem usada no treino e dentro do modelo 
✔ Medianas gravadas = medianas da tabela de auditoria do notebook 03m 
✔ Hiperparâmetros do modelo final = combinação escolhida (inclui class_weight='balanced') 
✔ O modelo aprendeu as classes 0..K-1, na ordem de classes.json 
✔ Ajuste e validação com as linhas planejad